# Load packages

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
import os
import glob
import re
import lingam
import networkx as nx
import requests
import shap
import warnings
import json
import dice_ml

from sklearn.preprocessing import (
    LabelEncoder,
    StandardScaler,
    MinMaxScaler,
)
from sklearn.metrics import (
    roc_auc_score, 
    roc_curve, 
    confusion_matrix, 
    precision_score, 
    recall_score, 
    accuracy_score,
    classification_report
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from typing import Dict
from sklearn.model_selection import cross_val_score, KFold
from sklearn.model_selection import train_test_split
from sklearn.model_selection import TimeSeriesSplit
from sklearn.utils.class_weight import compute_class_weight
from dotenv import load_dotenv
from collections import defaultdict
from IPython.display import display, Markdown
from datetime import datetime
from collections import Counter
from typing import Union

    

load_dotenv()

openai_api_key = os.getenv("OPENAI_API_KEY")
groq_api_key = os.getenv("GROQ_API_KEY")

warnings.filterwarnings('ignore')
print("✅ Imports loaded successfully!")


warnings.filterwarnings('ignore')

✅ Imports loaded successfully!


<br> <br> <br>

# Data Preprocessing

In [3]:
def rows_n_hours_before_failure(machine, machine_failure, hours):  
    failure_times = machine_failure["datetime"]

    # Convert the datetime columns to datetime if they're not already
    machine["datetime"] = pd.to_datetime(machine["datetime"])
    failure_times = pd.to_datetime(failure_times)

    # Initialize an empty list to store the rows and their indices
    rows = []
    indices = []

    # Iterate over each failure time and its index
    for idx_failure, failure_time in zip(machine_failure.index, failure_times):
        # Calculate the time n hours before the failure
        target_time = failure_time - pd.Timedelta(hours=hours)
        
        # Get the row with the closest time to the target time
        idx = (machine["datetime"] - target_time).abs().idxmin()
        closest_row = machine.loc[idx]
        
        # Append the row and the failure index to the lists
        rows.append(closest_row)
        indices.append(idx_failure)

    # Create a new dataframe with the rows
    machine_1_prev_24h = pd.DataFrame(rows)
    machine_1_prev_24h["id_failure_row"] = indices

    return machine_1_prev_24h


def update_lag_failure_target(machine_failure, machine_n_hours_before_failure, lag_failure):
    """
    Updates the 'target' variable in lag_failure with the corresponding 'failure' value from machine_failure,
    using machine_n_hours_before_failure as a bridge for the join.
    """
    if 'id_failure_row' not in machine_n_hours_before_failure.columns:
        raise ValueError("machine_n_hours_before_failure must have 'id_failure_row' column.")

    failure_values = []
    for idx in lag_failure.index:
        if idx in machine_n_hours_before_failure.index:
            id_failure_row = machine_n_hours_before_failure.loc[idx, 'id_failure_row']
            # If multiple rows, take the first
            if isinstance(id_failure_row, pd.Series):
                id_failure_row = id_failure_row.iloc[0]
            failure_val = machine_failure.loc[id_failure_row, 'failure'] if id_failure_row in machine_failure.index else None
        else:
            failure_val = None
        failure_values.append(failure_val)

    lag_failure = lag_failure.copy()
    lag_failure['target'] = failure_values
    return lag_failure


def create_safe_non_failure_samples_enhanced(machine_lag, lag_failure_updated, hours_before=24, safe_buffer_hours=48):
    """
    Create non-failure samples from 'safe zones' using the full feature set from machine_lag
    
    Parameters:
    - machine_lag: Full dataset with all engineered features
    - lag_failure_updated: Your target dataset (24h before failures)
    - hours_before: Lead time before failure (24h)
    - safe_buffer_hours: Additional buffer to ensure truly safe periods (48h recommended)
    """
    
    # Convert datetime columns to datetime if they aren't already
    machine_lag['datetime'] = pd.to_datetime(machine_lag['datetime'])
    lag_failure_updated['datetime'] = pd.to_datetime(lag_failure_updated['datetime'])
    
    # Get all failure timestamps from machine_lag where failure = 1
    failure_timestamps = machine_lag[machine_lag['failure'] == 1]['datetime']
    
    # Create exclusion zones around each failure
    exclusion_periods = []
    
    for failure_time in failure_timestamps:
        # Exclude from (failure_time - safe_buffer_hours) to failure_time
        start_exclusion = failure_time - pd.Timedelta(hours=safe_buffer_hours)
        end_exclusion = failure_time
        exclusion_periods.append((start_exclusion, end_exclusion))
    
    # Also exclude the timestamps that are already in lag_failure_updated (24h before failures)
    existing_target_timestamps = set(lag_failure_updated['datetime'])
    
    # Filter machine_lag to find safe timestamps
    safe_candidates = machine_lag.copy()
    
    # Remove rows that fall in exclusion zones
    def is_in_exclusion_zone(timestamp):
        for start_excl, end_excl in exclusion_periods:
            if start_excl <= timestamp <= end_excl:
                return True
        return False
    
    # Filter out exclusion zones and existing target timestamps
    safe_mask = (
        ~safe_candidates['datetime'].apply(is_in_exclusion_zone) &
        ~safe_candidates['datetime'].isin(existing_target_timestamps) &
        (safe_candidates['failure'] == 0)  # Only non-failure records
    )
    
    safe_candidates = safe_candidates[safe_mask]
    
    if len(safe_candidates) == 0:
        print("Warning: No safe candidates found. Consider reducing safe_buffer_hours.")
        return pd.DataFrame()
    
    # Sample non-failure records
    n_samples = min(len(lag_failure_updated), len(safe_candidates))
    
    if n_samples > len(safe_candidates):
        print(f"Warning: Only {len(safe_candidates)} safe candidates available, but {len(lag_failure_updated)} samples requested.")
        n_samples = len(safe_candidates)
    
    # Randomly sample from safe candidates
    non_failure_samples = safe_candidates.sample(n=n_samples, random_state=42).copy()
    
    # Set target to 0 for non-failure samples (they should already be 0, but ensure it)
    non_failure_samples['target'] = 0
    
    # Reset index
    non_failure_samples = non_failure_samples.reset_index(drop=True)
    
    return non_failure_samples

def create_balanced_dataset(machine_lag, lag_failure_updated, hours_before=24, safe_buffer_hours=48):
    """
    Create a balanced dataset combining failure predictions and safe non-failure samples
    """
    
    # Get non-failure samples
    non_failure_df = create_safe_non_failure_samples_enhanced(machine_lag, lag_failure_updated, hours_before, safe_buffer_hours)
    
    if len(non_failure_df) == 0:
        return lag_failure_updated.copy()
    
    # Ensure both dataframes have the same columns
    common_columns = list(set(lag_failure_updated.columns) & set(non_failure_df.columns))
    
    # If lag_failure_updated is missing some features, we need to merge them from machine_lag
    if len(common_columns) < len(machine_lag.columns):
        print("Merging additional features from machine_lag to lag_failure_updated...")
        
        # Merge lag_failure_updated with machine_lag to get all features
        lag_failure_enhanced = pd.merge(
            lag_failure_updated, 
            machine_lag, 
            on='datetime', 
            how='left',
            suffixes=('', '_from_machine_lag')
        )
        
        # Clean up duplicate columns (keep the original values from lag_failure_updated)
        for col in lag_failure_enhanced.columns:
            if col.endswith('_from_machine_lag'):
                original_col = col.replace('_from_machine_lag', '')
                if original_col in lag_failure_enhanced.columns:
                    lag_failure_enhanced[original_col] = lag_failure_enhanced[original_col].fillna(
                        lag_failure_enhanced[col]
                    )
                    lag_failure_enhanced = lag_failure_enhanced.drop(columns=[col])
        
        lag_failure_to_use = lag_failure_enhanced
    else:
        lag_failure_to_use = lag_failure_updated.copy()
    
    # Ensure all required columns are present in both dataframes
    all_required_columns = list(machine_lag.columns)
    
    # Add missing columns with appropriate default values if needed
    for col in all_required_columns:
        if col not in lag_failure_to_use.columns:
            lag_failure_to_use[col] = np.nan
        if col not in non_failure_df.columns:
            non_failure_df[col] = np.nan
    
    # Select only the required columns in the same order
    lag_failure_to_use = lag_failure_to_use[all_required_columns]
    non_failure_df = non_failure_df[all_required_columns]
    
    # Combine datasets
    balanced_dataset = pd.concat([lag_failure_to_use, non_failure_df], ignore_index=True)
    balanced_dataset = balanced_dataset.sort_values('datetime').reset_index(drop=True)
    
    return balanced_dataset, non_failure_df


# Alternative: If you want more control over sampling strategy
def create_stratified_non_failure_samples(machine_lag, lag_failure_updated, safe_buffer_hours=48, stratify_by=['comp', 'is_weekend', 'is_working_hours']):
    """
    Create stratified non-failure samples to ensure diversity across different conditions
    """
    
    machine_lag['datetime'] = pd.to_datetime(machine_lag['datetime'])
    lag_failure_updated['datetime'] = pd.to_datetime(lag_failure_updated['datetime'])
    
    # Get failure timestamps and create exclusion zones
    failure_timestamps = machine_lag[machine_lag['failure'] == 1]['datetime']
    exclusion_periods = []
    
    for failure_time in failure_timestamps:
        start_exclusion = failure_time - pd.Timedelta(hours=safe_buffer_hours)
        end_exclusion = failure_time
        exclusion_periods.append((start_exclusion, end_exclusion))
    
    # Filter safe candidates
    def is_in_exclusion_zone(timestamp):
        for start_excl, end_excl in exclusion_periods:
            if start_excl <= timestamp <= end_excl:
                return True
        return False
    
    existing_target_timestamps = set(lag_failure_updated['datetime'])
    
    safe_mask = (
        ~machine_lag['datetime'].apply(is_in_exclusion_zone) &
        ~machine_lag['datetime'].isin(existing_target_timestamps) &
        (machine_lag['failure'] == 0)
    )
    
    safe_candidates = machine_lag[safe_mask].copy()
    
    if len(safe_candidates) == 0:
        return pd.DataFrame()
    
    # Stratified sampling
    target_samples = len(lag_failure_updated)
    
    # Get the distribution of stratification variables in lag_failure_updated
    if all(col in lag_failure_updated.columns for col in stratify_by):
        # Sample proportionally to match the failure sample distribution
        stratified_samples = []
        
        for group_values, group_df in lag_failure_updated.groupby(stratify_by):
            group_size = len(group_df)
            proportion = group_size / len(lag_failure_updated)
            target_group_size = int(proportion * target_samples)
            
            # Find matching safe candidates
            mask = pd.Series(True, index=safe_candidates.index)
            for i, col in enumerate(stratify_by):
                mask &= (safe_candidates[col] == group_values[i])
            
            group_safe_candidates = safe_candidates[mask]
            
            if len(group_safe_candidates) > 0:
                sample_size = min(target_group_size, len(group_safe_candidates))
                group_sample = group_safe_candidates.sample(n=sample_size, random_state=42)
                stratified_samples.append(group_sample)
        
        if stratified_samples:
            non_failure_samples = pd.concat(stratified_samples, ignore_index=True)
        else:
            # Fallback to random sampling
            non_failure_samples = safe_candidates.sample(
                n=min(target_samples, len(safe_candidates)), 
                random_state=42
            )
    else:
        # Fallback to random sampling if stratification columns not available
        non_failure_samples = safe_candidates.sample(
            n=min(target_samples, len(safe_candidates)), 
            random_state=42
        )
    
    non_failure_samples['target'] = 0
    
    return non_failure_samples.reset_index(drop=True)

In [4]:
# Loop through machine_number 1 to 100 and generate balanced datasets for each machine

hours_before_failure = 1
safe_buffer_hours = 72

balanced_datasets = {}
non_failure_dfs = {}

for machine_number in range(1, 101):
    try:
        # Load data for this machine
        machine = pd.read_csv(f"../../data/azure_pm/machines/machine_{machine_number}.csv")
        machine_lag = pd.read_csv(f"../../data/azure_pm/lag_features/machine_{machine_number}_lag_features.csv")

        # Identify failures
        machine_failure = machine[machine['failure'] != '0']

        # Get rows n hours before each failure
        machine_n_hours_before_failure = rows_n_hours_before_failure(machine=machine,machine_failure=machine_failure,hours=hours_before_failure)
        lag_failure = rows_n_hours_before_failure(machine=machine_lag,machine_failure=machine_lag[machine_lag['failure'] != 0], hours=hours_before_failure)

        # Update lag_failure target
        lag_failure_updated = update_lag_failure_target(machine_failure=machine_failure, machine_n_hours_before_failure=machine_n_hours_before_failure, lag_failure=lag_failure)

        # Create balanced dataset
        balanced_dataset, non_failure_df = create_balanced_dataset(machine_lag=machine_lag, 
                                                                   lag_failure_updated=lag_failure_updated, 
                                                                   hours_before=hours_before_failure, 
                                                                   safe_buffer_hours=safe_buffer_hours)

        # Store results with machine number as key
        balanced_datasets[machine_number] = balanced_dataset
        non_failure_dfs[machine_number] = non_failure_df

        print(f"✅ Machine {machine_number}: Balanced dataset shape: {balanced_dataset.shape}, Non-failure shape: {non_failure_df.shape}")

    except Exception as e:
        print(f"❌ Machine {machine_number}: Error - {e}")

✅ Machine 1: Balanced dataset shape: (22, 68), Non-failure shape: (11, 68)
✅ Machine 2: Balanced dataset shape: (12, 68), Non-failure shape: (6, 68)
✅ Machine 3: Balanced dataset shape: (10, 68), Non-failure shape: (5, 68)
✅ Machine 4: Balanced dataset shape: (14, 68), Non-failure shape: (7, 68)
✅ Machine 5: Balanced dataset shape: (20, 68), Non-failure shape: (10, 68)
❌ Machine 6: Error - 'datetime'
✅ Machine 7: Balanced dataset shape: (34, 68), Non-failure shape: (17, 68)
✅ Machine 8: Balanced dataset shape: (10, 68), Non-failure shape: (5, 68)
✅ Machine 9: Balanced dataset shape: (32, 68), Non-failure shape: (16, 68)
✅ Machine 10: Balanced dataset shape: (16, 68), Non-failure shape: (8, 68)
✅ Machine 11: Balanced dataset shape: (18, 68), Non-failure shape: (9, 68)
✅ Machine 12: Balanced dataset shape: (18, 68), Non-failure shape: (9, 68)
✅ Machine 13: Balanced dataset shape: (44, 68), Non-failure shape: (22, 68)
✅ Machine 14: Balanced dataset shape: (14, 68), Non-failure shape: (7, 

In [5]:
# Join all balanced_datasets into a single DataFrame named df_union
df_union = pd.concat(
    [df for df in balanced_datasets.values() if not df.empty],
    ignore_index=True
)

# Optional: sort by datetime and reset index for consistency
df_union = df_union.sort_values(by='datetime').reset_index(drop=True)

df = df_union.copy()

# Apply mapping to target column so it's always integer
target_mapping = {
    "0": 0,
    0: 0,
    "comp1": 1,
    1: 1,
    "comp2": 2,
    2: 2,
    "comp3": 3,
    3: 3,
    "comp4": 4,
    4: 4
}
df["target"] = df["target"].map(target_mapping).astype(int)

In [6]:
df.head()

,datetime,machineID,volt,rotate,pressure,vibration,model,age,errorID,comp,...,error_count_6h,error_count_24h,maint_count_6h,maint_count_24h,hour,day_of_week,is_weekend,is_working_hours,hours_since_maint,hours_since_error
0,2015-01-01 20:00:00,21,0.510555,0.319933,0.376847,0.355634,model2,14,0,0,...,0.0,0.0,0.0,0.0,20,3,0,0,14,14
1,2015-01-01 20:00:00,85,0.525555,0.489049,0.507522,0.719747,model1,16,0,0,...,0.0,0.0,0.0,0.0,20,3,0,0,14,14
2,2015-01-01 20:00:00,23,0.644100,0.464922,0.459205,0.392635,model1,17,0,0,...,0.0,0.0,0.0,0.0,20,3,0,0,14,14
3,2015-01-01 20:00:00,37,0.524530,0.532051,0.351415,0.465700,model1,16,0,0,...,0.0,0.0,0.0,0.0,20,3,0,0,14,14
4,2015-01-02 02:00:00,22,0.716302,0.628508,0.380510,0.407093,model1,14,0,0,...,0.0,0.0,0.0,0.0,2,4,0,0,20,20


In [7]:
df.columns

Index(['datetime', 'machineID', 'volt', 'rotate', 'pressure', 'vibration',
       'model', 'age', 'errorID', 'comp', 'failure', 'target', 'volt_lag_1h',
       'volt_lag_6h', 'volt_lag_12h', 'volt_lag_24h', 'volt_mean_24h',
       'volt_std_24h', 'volt_min_24h', 'volt_max_24h', 'volt_mean_6h',
       'volt_std_6h', 'rotate_lag_1h', 'rotate_lag_6h', 'rotate_lag_12h',
       'rotate_lag_24h', 'rotate_mean_24h', 'rotate_std_24h', 'rotate_min_24h',
       'rotate_max_24h', 'rotate_mean_6h', 'rotate_std_6h', 'pressure_lag_1h',
       'pressure_lag_6h', 'pressure_lag_12h', 'pressure_lag_24h',
       'pressure_mean_24h', 'pressure_std_24h', 'pressure_min_24h',
       'pressure_max_24h', 'pressure_mean_6h', 'pressure_std_6h',
       'vibration_lag_1h', 'vibration_lag_6h', 'vibration_lag_12h',
       'vibration_lag_24h', 'vibration_mean_24h', 'vibration_std_24h',
       'vibration_min_24h', 'vibration_max_24h', 'vibration_mean_6h',
       'vibration_std_6h', 'errorID_lag_1h', 'errorID_lag_6h',


<br> <br> <br>

Remove columns

In [ ]:
df.drop(columns=['comp_lag_1h',
                 'comp_lag_6h', 
                 'comp_lag_12h', 
                 'maint_count_6h', 
                 'maint_count_24h', 
                 'hour', 
                 'day_of_week', 
                 'is_weekend', 
                 'is_working_hours', 
                 'hours_since_maint', 
                 'hours_since_error'
                 ], inplace=True)


# df.columns

In [ ]:
df.head()

<br> <br> <br>

---

<br> <br>

<br> <br> <br>

# Machine Learning

In [ ]:
# Convert df["model"] to numeric in-place.
# Strategy: try direct numeric conversion -> extract trailing digits -> fallback to LabelEncoder.

try:
    df['model'] = pd.to_numeric(df['model'])
except Exception:
    digits = df['model'].astype(str).str.extract(r'(\d+)')
    if not digits.isnull().values.any():
        df['model'] = digits[0].astype(int)
    else:
        le = LabelEncoder()
        df['model'] = le.fit_transform(df['model'].astype(str))

print("Unique model values after conversion:", df['model'].unique())

In [ ]:
df["target"]

In [ ]:
class MultiClassPredictiveMaintenanceEvaluator:
    """
    Comprehensive evaluator for multi-class predictive maintenance models.
    """

    def __init__(self, random_state=42):
        self.random_state = random_state
        self.models = {}
        self.results = {}

    def prepare_data(self, df, test_size=0.2, target_col='target', excluded_column = ['datetime', 'failure', 'target']):
        """
        Prepare and split data, ensuring temporal sequence and correct data formats.
        """
        df['datetime'] = pd.to_datetime(df['datetime'])
        df = df.sort_values(by='datetime').reset_index(drop=True)

        # exclude_cols = ['datetime', 'failure', 'target']
        exclude_cols = excluded_column
        feature_cols = [col for col in df.columns if col not in exclude_cols]
        X = df[feature_cols].fillna(0)
        y = df[target_col]

        split_index = int(len(df) * (1 - test_size))
        X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
        y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

        # Convert y_train and y_test to 1D NumPy arrays to ensure compatibility
        y_train = y_train.values
        y_test = y_test.values

        print(f"Dataset shape: {df.shape}")
        print(f"Feature columns: {len(feature_cols)}")
        print(f"\nTrain set shape: {X_train.shape}")
        print(f"Test set shape: {X_test.shape}\n")

        return X_train, X_test, y_train, y_test

    def initialize_models(self):
        """
        Initialize all machine learning models.
        """
        self.models = {
            'RandomForest': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=self.random_state, n_jobs=-1),
            'XGBoost': XGBClassifier(n_estimators=100, max_depth=6, random_state=self.random_state, eval_metric='mlogloss'),
            'LightGBM': LGBMClassifier(n_estimators=100, max_depth=6, random_state=self.random_state, verbose=-1),
            'CatBoost': CatBoostClassifier(iterations=100, depth=8, learning_rate=0.03,l2_leaf_reg=5, auto_class_weights="Balanced" ,random_seed=self.random_state, verbose=100)
        }

    def calculate_metrics(self, y_true, y_pred, y_pred_proba):
        """
        Calculate comprehensive metrics for multi-class classification.
        """
        metrics = {}
        classes = np.unique(y_true)
        
        # 1. ROC AUC (macro and micro)
        metrics['ROC_AUC_macro'] = roc_auc_score(y_true, y_pred_proba, multi_class='ovr', average='macro')
        metrics['ROC_AUC_micro'] = roc_auc_score(y_true, y_pred_proba, multi_class='ovr', average='micro')

        # 2. Normalized AUC
        per_class_aucs = [roc_auc_score(y_true == c, y_pred_proba[:, i]) for i, c in enumerate(classes)]
        metrics['Normalized_AUC'] = np.mean([(auc - 0.5) / 0.5 for auc in per_class_aucs])

        # 3. Youden Index
        youden_scores = []
        for i, c in enumerate(classes):
            y_true_binary = (y_true == c)
            y_pred_binary = (y_pred == c)
            tn, fp, fn, tp = confusion_matrix(y_true_binary, y_pred_binary).ravel()
            sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
            specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
            youden_scores.append(sensitivity + specificity - 1)
        metrics['Youden_Index'] = np.mean(youden_scores)
        
        # 4. Precision (macro and micro)
        metrics['Precision_macro'] = precision_score(y_true, y_pred, average='macro', zero_division=0)
        metrics['Precision_micro'] = precision_score(y_true, y_pred, average='micro', zero_division=0)

        # 5. Recall (macro and micro)
        metrics['Recall_macro'] = recall_score(y_true, y_pred, average='macro', zero_division=0)
        metrics['Recall_micro'] = recall_score(y_true, y_pred, average='micro', zero_division=0)
        
        metrics['Accuracy'] = np.mean(y_true == y_pred)
        return metrics

    def train_and_evaluate(self, X_train, X_test, y_train, y_test):
        """
        Train all models and evaluate their performance.
        """
        X_train.columns = [re.sub(r'[^A-Za-z0-9_]+', '', str(col)) for col in X_train.columns]
        X_test.columns = [re.sub(r'[^A-Za-z0-9_]+', '', str(col)) for col in X_test.columns]

        self.initialize_models()
        self.results = {}

        for model_name, model in self.models.items():
            print(f"Training {model_name}...")
            model.fit(X_train, y_train)

            y_pred = model.predict(X_test)
            y_pred_proba = model.predict_proba(X_test)
            metrics = self.calculate_metrics(y_test, y_pred, y_pred_proba)

            self.results[model_name] = {'metrics': metrics, 'y_pred': y_pred}
            print(f"{model_name} - ROC AUC (macro): {metrics['ROC_AUC_macro']:.4f}, Accuracy: {metrics['Accuracy']:.4f}")

    def print_detailed_results(self, y_test):
        """
        Print detailed classification reports and confusion matrices.
        """
        print("\n" + "="*80 + "\nDETAILED EVALUATION RESULTS\n" + "="*80)
        summary_data = [{'Model': name, **{k: f"{v:.4f}" for k, v in res['metrics'].items()}}
                        for name, res in self.results.items()]
        
        print("\nSUMMARY OF ALL METRICS:")
        print(pd.DataFrame(summary_data).to_string(index=False))

        for model_name, result in self.results.items():
            print(f"\n{'-'*60}\nDETAILED RESULTS FOR {model_name}\n{'-'*60}")
            print("Classification Report:\n", classification_report(y_test, result['y_pred']))
            print("Confusion Matrix:\n", confusion_matrix(y_test, result['y_pred']))

    def plot_results(self, figsize=(15, 10)):
        """
        Create and display visualizations of model performance.
        """
        fig, axes = plt.subplots(2, 3, figsize=figsize)
        fig.suptitle('Model Performance Comparison', fontsize=16)
        axes = axes.ravel()
        
        metrics_to_plot = ['ROC_AUC_macro', 'Precision_macro', 'Recall_macro', 'Normalized_AUC', 'Youden_Index', 'Accuracy']
        model_names = list(self.results.keys())

        for i, metric in enumerate(metrics_to_plot):
            ax = axes[i]
            scores = [self.results[model]['metrics'][metric] for model in model_names]
            ax.bar(model_names, scores, color=plt.cm.viridis(np.linspace(0, 1, len(model_names))))
            ax.set_title(metric.replace('_', ' ').title())
            ax.tick_params(axis='x', labelrotation=45)
            ax.grid(axis='y', linestyle='--', alpha=0.7)

        plt.tight_layout(rect=[0, 0.03, 1, 0.95])
        plt.show()

    
    def calculate_metrics(self, y_true, y_pred, y_pred_proba):
        """
        Calculate comprehensive metrics for multi-class classification.
        Also compute binary (one-vs-rest) metrics for class '2' when available:
          - ROC AUC (binary for class 2)
          - Normalized AUC (binary for class 2)
          - Youden Index (binary for class 2)
          - Precision / Recall (binary for class 2)
        """
        metrics = {}
        classes = np.unique(y_true)

        # Multi-class ROC AUC (OVR) — guard with try/except
        try:
            metrics['ROC_AUC_macro'] = roc_auc_score(y_true, y_pred_proba, multi_class='ovr', average='macro')
            metrics['ROC_AUC_micro'] = roc_auc_score(y_true, y_pred_proba, multi_class='ovr', average='micro')
        except Exception:
            metrics['ROC_AUC_macro'] = np.nan
            metrics['ROC_AUC_micro'] = np.nan

        # Per-class AUCs (safe)
        per_class_aucs = []
        for i, c in enumerate(classes):
            try:
                auc_val = roc_auc_score((y_true == c).astype(int), y_pred_proba[:, i])
            except Exception:
                auc_val = np.nan
            per_class_aucs.append(auc_val)

        # Normalized AUC: mean of (auc - 0.5)/0.5 across available per-class AUCs
        normalized_values = [(auc - 0.5) / 0.5 for auc in per_class_aucs if not np.isnan(auc)]
        metrics['Normalized_AUC'] = np.nanmean(normalized_values) if normalized_values else np.nan

        # Youden Index per class (safe)
        youden_scores = []
        for i, c in enumerate(classes):
            try:
                y_true_bin = (y_true == c).astype(int)
                y_pred_bin = (y_pred == c).astype(int)
                tn, fp, fn, tp = confusion_matrix(y_true_bin, y_pred_bin).ravel()
                sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
                specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
                youden_scores.append(sensitivity + specificity - 1)
            except Exception:
                youden_scores.append(np.nan)
        valid_youden = [s for s in youden_scores if not np.isnan(s)]
        metrics['Youden_Index'] = np.nanmean(valid_youden) if valid_youden else np.nan

        # Precision & Recall (macro & micro)
        metrics['Precision_macro'] = precision_score(y_true, y_pred, average='macro', zero_division=0)
        metrics['Precision_micro'] = precision_score(y_true, y_pred, average='micro', zero_division=0)
        metrics['Recall_macro'] = recall_score(y_true, y_pred, average='macro', zero_division=0)
        metrics['Recall_micro'] = recall_score(y_true, y_pred, average='micro', zero_division=0)

        # Accuracy
        metrics['Accuracy'] = np.mean(y_true == y_pred)

        # Binary one-vs-rest metrics specifically for class '2' (if present)
        class_of_interest = 2
        if class_of_interest in classes:
            # find index of class 2 in classes ordering (aligns with y_pred_proba columns)
            class_index = list(classes).index(class_of_interest)
            y_true_bin = (y_true == class_of_interest).astype(int)
            y_pred_bin = (y_pred == class_of_interest).astype(int)

            # ROC AUC for class 2 (binary), normalized AUC
            try:
                auc_class2 = roc_auc_score(y_true_bin, y_pred_proba[:, class_index])
                metrics['ROC_AUC_class2'] = auc_class2
                metrics['Normalized_AUC_class2'] = (auc_class2 - 0.5) / 0.5
            except Exception:
                metrics['ROC_AUC_class2'] = np.nan
                metrics['Normalized_AUC_class2'] = np.nan

            # Youden Index for class 2
            try:
                tn, fp, fn, tp = confusion_matrix(y_true_bin, y_pred_bin).ravel()
                sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
                specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
                metrics['Youden_Index_class2'] = sensitivity + specificity - 1
            except Exception:
                metrics['Youden_Index_class2'] = np.nan

            # Precision / Recall for class 2 (binary)
            metrics['Precision_class2'] = precision_score(y_true_bin, y_pred_bin, zero_division=0)
            metrics['Recall_class2'] = recall_score(y_true_bin, y_pred_bin, zero_division=0)
        else:
            # Fill keys with NaN when class 2 not present
            metrics['ROC_AUC_class2'] = np.nan
            metrics['Normalized_AUC_class2'] = np.nan
            metrics['Youden_Index_class2'] = np.nan
            metrics['Precision_class2'] = np.nan
            metrics['Recall_class2'] = np.nan

        return metrics




excluded_columns = ['datetime', 'failure', 'target'] 


def run_complete_evaluation(df):
    """
    Run the complete evaluation pipeline.
    """
    evaluator = MultiClassPredictiveMaintenanceEvaluator(random_state=42)
    X_train, X_test, y_train, y_test = evaluator.prepare_data(df=df, excluded_column = excluded_columns)
    evaluator.train_and_evaluate(X_train, X_test, y_train, y_test)
    evaluator.print_detailed_results(y_test)
    evaluator.plot_results()
    return evaluator, X_test, y_test, X_train, y_train

# --- Main Execution ---
# df = pd.read_csv("../../data/azure_pm/balanced_dataset_safezone/merged_dataset/balanced_azure_pm.csv")
evaluator_results, X_test, y_test, X_train, y_train = run_complete_evaluation(df)

In [ ]:
# 1) get a model (from evaluator instance)
model = evaluator_results.models['RandomForest']  

# 2) compute predictions on X_test
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)   # shape: (n_samples, n_classes)

# 3) call calculate_metrics
metrics = evaluator_results.calculate_metrics(y_test, y_pred, y_pred_proba)

# 4) view results
metrics

In [ ]:
# 1) get a model (from evaluator instance)
model = evaluator_results.models['XGBoost']   # or 'RandomForest','LightGBM','CatBoost'

# 2) compute predictions on X_test
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)   # shape: (n_samples, n_classes)

# 3) call calculate_metrics
metrics = evaluator_results.calculate_metrics(y_test, y_pred, y_pred_proba)

# 4) view results
metrics

In [ ]:
# 1) get a model (from evaluator instance)
model = evaluator_results.models['LightGBM']  

# 2) compute predictions on X_test
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)   # shape: (n_samples, n_classes)

# 3) call calculate_metrics
metrics = evaluator_results.calculate_metrics(y_test, y_pred, y_pred_proba)

# 4) view results
metrics

In [ ]:
# 1) get a model (from evaluator instance)
model = evaluator_results.models['CatBoost']  

# 2) compute predictions on X_test
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)   # shape: (n_samples, n_classes)

# 3) call calculate_metrics
metrics = evaluator_results.calculate_metrics(y_test, y_pred, y_pred_proba)

# 4) view results
metrics

<br> <br>

CatBoost accuracy is much lower than other ML models because of one or more of the following reasons:

- **Class imbalance handling:** CatBoost may not be configured with the correct class weights or balancing strategy for your multi-class, imbalanced dataset. The other models (RandomForest, LightGBM, XGBoost) have explicit parameters for class weighting (`class_weight`, `scale_pos_weight`, etc.), but CatBoost requires `class_weights` or `auto_class_weights='Balanced'` for proper balancing.
- **Hyperparameter settings:** The default hyperparameters for CatBoost (e.g., `iterations`, `depth`, `learning_rate`) may not be optimal for your data, while the other models might be better tuned.
- **Feature preprocessing:** CatBoost handles categorical features natively, but if your features are already encoded or not properly specified, it may not leverage its strengths.
- **Random seed or data split:** If the random seed or train/test split is not consistent, CatBoost may be affected more due to its internal handling of randomness.
- **Implementation bug:** There could be a bug in how CatBoost is initialized or trained in your pipeline. For example, if the target labels are not correctly mapped or if the training data is not properly shuffled.

To improve CatBoost accuracy, check:
- That you set `auto_class_weights='Balanced'` or provide explicit `class_weights` for imbalanced data.
- That your target labels are correctly mapped and encoded.
- That you tune CatBoost hyperparameters.
- That you use the same train/test split and preprocessing for all models.

You can review the CatBoost initialization in your code at `MultiClassPredictiveMaintenanceEvaluator` for possible improvements.

In [ ]:
# Save each model as a .pkl file

# model_save_path = "../../model/azure_pm/balanced_model/"
model_save_path = "models/"

for model_name, model in evaluator_results.models.items():
    with open(f"{model_save_path}{model_name}.pkl", "wb") as f:
          pickle.dump(model, f)          

<br> <br>

---

<br> <br> <br>


# Causality

In [ ]:
def create_causal_network(df, features, regularize=True, noise_level=1e-4, self_reference=True):
    """
    Build a causal network from VARLiNGAM adjacency matrices.

    Args:
      df: dataframe with data
      features: list of feature names (order must match columns used to fit VARLiNGAM)
      regularize: if True, add small noise on LinAlgError and retry
      noise_level: std of noise to add when regularizing
      self_reference: if False, remove/self-ignore edges where source == target (no self-loops)

    Returns:
      structured_adjacency: list of dicts {source_feature, target_feature, effect_strength}
      fig: matplotlib figure with plotted network
    """
    X_selected = df[features].values

    # Initialize the VARLiNGAM model
    model = lingam.VARLiNGAM()

    # Attempt to fit the model; add noise if a LinAlgError occurs
    try:
        model.fit(X_selected)
    except np.linalg.LinAlgError as e:
        print("LinAlgError encountered during model fitting:", e)
        if regularize:
            print(f"Applying regularization: adding noise (std={noise_level}) to the data and trying again.")
            X_selected += np.random.normal(0, noise_level, X_selected.shape)
            model.fit(X_selected)
        else:
            raise e

    # Retrieve the adjacency matrices (one per lag)
    adjacency_matrices = model.adjacency_matrices_

    # Build a directed graph and create a list of dictionaries for each non-zero edge.
    G = nx.DiGraph()
    G.add_nodes_from(features)
    n_features = len(features)
    structured_adjacency = []

    # Loop through each adjacency matrix (one per lag)
    for lag_index, adj_matrix in enumerate(adjacency_matrices):
        for i in range(n_features):
            for j in range(n_features):
                # skip self-references if requested
                if (not self_reference) and (i == j):
                    continue
                weight = adj_matrix[i, j]
                if weight != 0:
                    edge_dict = {
                        "source_feature": features[i],
                        "target_feature": features[j],
                        "effect_strength": weight,
                    }
                    structured_adjacency.append(edge_dict)
                    G.add_edge(features[i], features[j], weight=weight)

    # Plot the graph using a spring layout for clarity
    pos = nx.spring_layout(G, seed=42)  # seed for reproducibility
    fig, ax = plt.subplots(figsize=(12, 8))
    nx.draw(
        G,
        pos,
        with_labels=True,
        node_color="lightblue",
        node_size=1500,
        arrowstyle="->",
        arrowsize=20,
        edge_color="gray",
        font_size=10,
        ax=ax,
    )

    # Display the edge weights formatted to two decimal places
    edge_labels = nx.get_edge_attributes(G, "weight")
    edge_labels = {edge: f"{weight:.2f}" for edge, weight in edge_labels.items()}
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_color="red", ax=ax)

    title_suffix = " (no self-references)" if not self_reference else ""
    ax.set_title("Causal Network Graph from VARLiNGAM" + title_suffix)
    ax.axis("off")
    # plt.show()
    plt.close()

    return structured_adjacency, fig


In [ ]:
cols_without_datetime_failure = [col for col in df.columns if "datetime" not in col and "failure" not in col]


# structured_adjacency, fig = create_causal_network(df=df, features=cols_without_datetime_failure)
structured_adjacency, fig = create_causal_network(df=df, features=cols_without_datetime_failure, self_reference=False)

In [ ]:
display(fig)

---

<br> <br> 

# Generate top-n rows using SHAP

#### Read ML models

In [ ]:
model_dir = "models/"
pkl_files = glob.glob(os.path.join(model_dir, "*.pkl"))

models = {}
for file_path in pkl_files:
    model_name = os.path.splitext(os.path.basename(file_path))[0]
    with open(file_path, "rb") as f:
        models[model_name] = pickle.load(f)

print(f"Loaded {len(models)} models: {list(models.keys())}")

my_models = {
    "catboost_model": 'CatBoost', 
    "lightgbm_model": 'LightGBM', 
    "randomforest_model": 'RandomForest', 
    "xgboost_model": 'XGBoost'
}

# model = models["XGBoost"]
# model = models[my_models["catboost_model"]]
# model = models[my_models["lightgbm_model"]]
# model = models[my_models["randomforest_model"]]
model = models[my_models["xgboost_model"]]
model



<br> <br> <br>

## Find Uncertain Rows

In [ ]:
X_test
# y_test

In [ ]:
def get_uncertain_rows(X_test, y_test, model, uncertainty_threshold=0.6):
    """
    Returns rows from X_test where the model's prediction confidence is below the uncertainty_threshold
    and the true target is not 0. Also prints the predicted class, its probability, and the true target value for each uncertain row.

    Parameters:
        X_test (pd.DataFrame): Test features.
        y_test (array-like): True labels for X_test.
        model: Trained ML model with predict_proba method.
        uncertainty_threshold (float): Maximum probability threshold for uncertainty.

    Returns:
        pd.DataFrame: Uncertain rows from X_test with their true and predicted labels and probabilities.
    """
    probs = model.predict_proba(X_test)
    max_probs = probs.max(axis=1)
    preds = model.classes_[probs.argmax(axis=1)]
    mask = (max_probs < uncertainty_threshold) & (y_test != 0)

    uncertain_rows = X_test[mask].copy()
    uncertain_rows['true_label'] = y_test[mask]
    uncertain_rows['predicted_label'] = preds[mask]
    uncertain_rows['max_probability'] = max_probs[mask]
    uncertain_rows['all_probabilities'] = list(probs[mask])

    # Print predicted class, probability, and true target value for each uncertain row
    for idx, row in uncertain_rows.iterrows():
        print(
            f"Index: {idx}, Predicted class: {row['predicted_label']}, "
            f"Probability: {row['max_probability']:.4f}, True target value: {row['true_label']}"
        )

    return uncertain_rows

# Example usage:
uncertain_rows = get_uncertain_rows(X_test=X_test, y_test=y_test, model=model, uncertainty_threshold=0.6)
display(uncertain_rows)



In [ ]:
row_number = 1853


random_row = X_test.loc[row_number].to_frame().T
random_row_target_label = uncertain_rows.loc[row_number]["true_label"]
print(f"The true label is: {uncertain_rows.loc[row_number]['true_label']}, and the model predicted {uncertain_rows.loc[row_number]['predicted_label']}")

display(random_row)

# random_row.drop(columns=["comp_lag_12h"], inplace=True)

<br> <br> 

## Generate top-n rows using SHAP

In [ ]:
def get_top_n_shap_features(random_row, model, top_n=10):
    feature_names = model.feature_names_in_
    X_row = random_row
    X_row = X_row.apply(pd.to_numeric, errors='coerce')

    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_row)

    if isinstance(shap_values, list):
        shap_abs = np.sum([np.abs(sv).reshape(-1) for sv in shap_values], axis=0)
    else:
        shap_abs = np.abs(shap_values).reshape(-1)

    min_len = min(len(feature_names), len(shap_abs))
    df_shap = pd.DataFrame({
        'feature_name': feature_names[:min_len],
        'feature_importance': shap_abs[:min_len]
    }).sort_values('feature_importance', ascending=False).head(top_n).reset_index(drop=True)

    return df_shap


df_shap = get_top_n_shap_features(random_row=random_row, model=model, top_n=15)
display(df_shap)


# Visualize SHAP values from df_shap as a horizontal bar plot
plt.figure(figsize=(8, 6))
plt.barh(df_shap['feature_name'][::-1], df_shap['feature_importance'][::-1], color='skyblue')
plt.xlabel('SHAP Value (absolute)')
plt.title('Top SHAP Feature Importances')
plt.show()

<br> <br>

---

<br> <br> <br>

# Taxonomy  

<br> <br> <br>

## Generate Taxonomy

In [11]:
# ```python
def generate_taxonomy_with_groq(columns_list, api_key, model="llama3-70b-8192", output_file="groq_taxonomy.json"):
    output_file = f"{output_file}"
    headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}

    # API endpoint
    endpoint = "https://api.groq.com/openai/v1/chat/completions"

    # Create the prompt for the taxonomy
    prompt = f"""
    **Task**:  
    Create a hierarchical taxonomy for dataset features.

    **Instructions**:
    
    1. **Organize Features Hierarchically**:  
    - Group dataset features based on their semantic meaning and relationships.
    - The taxonomy should show clear parent-child relationships between features.

    2. **Output Must Be Valid JSON**:
    - Ensure the response is in **JSON format**.
    - No extra text or explanations.

    **Example Format**:
    {{
        "Type": [1, "Root"],
        "Temperature": [1, "Root"],
        "Air_temperature": [2, "Temperature"]
        "Air_temperature_combination": [3, "Air_temperature"]
        
    }}

    **Input Features**:
    {', '.join(columns_list)}
    """

    # Create the request payload
    payload = {
        "model": model,  # Using the model specified as parameter
        "messages": [
            {
                "role": "system",
                "content": "You are a data scientist who organizes features into taxonomies.",
            },
            {"role": "user", "content": prompt},
        ],
        "temperature": 0.2,
    }

    # Make the API call
    response = requests.post(endpoint, headers=headers, json=payload)

    if response.status_code == 200:
        result = response.json()
        content = result["choices"][0]["message"]["content"]

        # Extract JSON from the response
        json_match = re.search(r"{.*}", content, re.DOTALL)
        if json_match:
            taxonomy_json_str = json_match.group(0)
            taxonomy = json.loads(taxonomy_json_str)
        else:
            # If no JSON pattern found, try to parse the whole content
            try:
                taxonomy = json.loads(content)
            except:
                raise ValueError("Failed to parse JSON from API response")

        # Save to file
        with open(output_file, "w") as f:
            json.dump(taxonomy, f, indent=2)

        return taxonomy
    else:
        error_message = f"API request failed with status code {response.status_code}"
        try:
            error_details = response.json()
            error_message += f": {error_details}"
        except:
            pass
        raise Exception(error_message)


def generate_taxonomy_with_openai(filename, features, openai_api_key):
    prompt = f"""
    **Task**:  
    Create a hierarchical taxonomy for dataset features.

    **Instructions**:
    
    1. **Organize Features Hierarchically**:  
    - Group dataset features based on their semantic meaning and relationships.
    - The taxonomy should show clear parent-child relationships between features.

    2. **Output Must Be Valid JSON**:
    - Ensure the response is in **JSON format**.
    - No extra text or explanations.

    3. The hierarchy can be at any levels (e.g. 2,3,4,etc) but should be logical and meaningful.

    **Example Format**:
    {{
        "Type": [1, "Root"],
        "Temperature": [1, "Root"],
        "Air_temperature": [2, "Temperature"]

    }}

    **Input Features**:
    {json.dumps(features, indent=2)}
    """

    openai_url = "https://api.openai.com/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {openai_api_key}",
        "Content-Type": "application/json",
    }
    data = {
        "model": "gpt-4o",
        "messages": [
            {
                "role": "system",
                "content": "Respond only with valid JSON, no extra text.",
            },
            {"role": "user", "content": prompt},
        ],
        "response_format": {
            "type": "json_object"
        },  # ✅ Changed from "json" to "json_object"
        "temperature": 0,
    }

    response = requests.post(openai_url, headers=headers, json=data)

    if response.status_code == 200:
        raw_content = response.json()["choices"][0]["message"]["content"].strip()

        # print("🔍 Raw OpenAI Response:\n", raw_content)  # Debugging step

        # Extract only JSON if extra text is present
        json_match = re.search(r"\{.*\}", raw_content, re.DOTALL)
        if json_match:
            raw_content = json_match.group(0)  # Extract JSON only

        # Try to parse as JSON
        try:
            taxonomy = json.loads(raw_content)
        except json.JSONDecodeError:
            raise RuntimeError(
                f"OpenAI API returned an invalid JSON response:\n{raw_content}"
            )

        # Save taxonomy to a file
        with open(f"{filename}", "w") as f:
            json.dump(taxonomy, f, indent=2)

        return taxonomy
    else:
        raise RuntimeError(
            f"Error from OpenAI API: {response.status_code} {response.text}"
        )

def generate_taxonomy_with_openai_edited(filename, features, root_features, openai_api_key):
    prompt = f"""
    **Task**:  
    Create a hierarchical taxonomy for dataset features.

    **Instructions**:
    
    1. **Organize Features Hierarchically**:  
    - Group dataset features based on their semantic meaning and relationships.
    - The taxonomy should show clear parent-child relationships between features.
    - The hierachy MUST NOT include features that are not existed in the dataset's features  
      Dataset's list of feature: {features} 
    -The roots of the hierarchy are {root_features}. These features can not have parents.


    2. **Output Must Be Valid JSON**:
    - Ensure the response is in **JSON format**.
    - No extra text or explanations.

    3. The hierarchy can be at any levels (e.g. 2,3,4,etc) but should be logical and meaningful.

    **Example Format**:
    {{
        "Type": [1, "Root"],
        "Temperature": [1, "Root"],
        "Air_temperature": [2, "Temperature"]

    }}

    **Input Features**:
    {json.dumps(features, indent=2)}
    """

    openai_url = "https://api.openai.com/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {openai_api_key}",
        "Content-Type": "application/json",
    }
    data = {
        "model": "gpt-4o",
        "messages": [
            {
                "role": "system",
                "content": "Respond only with valid JSON, no extra text.",
            },
            {"role": "user", "content": prompt},
        ],
        "response_format": {
            "type": "json_object"
        },  # ✅ Changed from "json" to "json_object"
        "temperature": 0,
    }

    response = requests.post(openai_url, headers=headers, json=data)

    if response.status_code == 200:
        raw_content = response.json()["choices"][0]["message"]["content"].strip()

        # print("🔍 Raw OpenAI Response:\n", raw_content)  # Debugging step

        # Extract only JSON if extra text is present
        json_match = re.search(r"\{.*\}", raw_content, re.DOTALL)
        if json_match:
            raw_content = json_match.group(0)  # Extract JSON only

        # Try to parse as JSON
        try:
            taxonomy = json.loads(raw_content)
        except json.JSONDecodeError:
            raise RuntimeError(
                f"OpenAI API returned an invalid JSON response:\n{raw_content}"
            )

        # Save taxonomy to a file
        with open(f"{filename}", "w") as f:
            json.dump(taxonomy, f, indent=2)

        return taxonomy
    else:
        raise RuntimeError(
            f"Error from OpenAI API: {response.status_code} {response.text}"
        )
# ```

<br> <br> <br>

### Generate the taxonomy


In [12]:
# ```python
llama3_taxonomy = generate_taxonomy_with_groq(columns_list=df.columns, api_key=groq_api_key, model="llama-3.3-70b-versatile", output_file="taxonomy/llama3.json")

gemma2_taxonomy = generate_taxonomy_with_groq(columns_list=df.columns, api_key=groq_api_key, model="gemma2-9b-it", output_file="taxonomy/gemma2.json")

deepseek_taxonomy = generate_taxonomy_with_groq(columns_list=df.columns, api_key=groq_api_key, model="deepseek-r1-distill-llama-70b", output_file="taxonomy/deepseek.json")

gpt4o_taxonomy = generate_taxonomy_with_openai(filename="taxonomy/gpt4o.json", features=df.columns.tolist(), openai_api_key=openai_api_key)

root_features = ["datetime", "volt", "rotate", "pressure", "vibration", "errorID", "comp", "failure"]
gpt4o_edited_taxonomy = generate_taxonomy_with_openai_edited(filename="taxonomy/gpt4o_edited.json", 
                                                             features=df.columns.tolist(), 
                                                             root_features = root_features, 
                                                             openai_api_key=openai_api_key)
# ```

<br> <br> <br>

## Load the Taxonomy 

In [8]:
with open('taxonomy/deepseek.json', 'r') as file:
    deepseek_taxonomy = json.load(file)


with open('taxonomy/gemma2.json', 'r') as file:
    gemma2_taxonomy = json.load(file)


with open('taxonomy/llama3.json', 'r') as file:
    llama3_taxonomy = json.load(file)


with open('taxonomy/gpt4o.json', 'r') as file:
    gpt4o_taxonomy = json.load(file)


with open('taxonomy/gpt4o_edited.json', 'r') as file:
    gpt4o_edited_taxonomy = json.load(file)    

In [9]:
def get_taxonomy_paths(taxonomy_json, node_names):
    
    """Returns hierarchical paths for given nodes in a taxonomy."""
    if isinstance(taxonomy_json, str):
        taxonomy_json = json.loads(taxonomy_json)  # Parse JSON if it's a string

    # Reverse lookup: Map each child to its parent
    child_to_parent = {child: parent for child, (_, parent) in taxonomy_json.items()}

    def get_path(node):
        """Recursively constructs the hierarchy path for a node."""
        path = [node]
        while node in child_to_parent and child_to_parent[node] != "Root":
            node = child_to_parent[node]
            path.append(node)
        return " --> ".join(reversed(path))  # Reverse to get correct hierarchy order
    return {node: get_path(node) for node in node_names}



def format_taxonomy(taxonomy_json):
    """Formats a hierarchical taxonomy into a tree-like string."""
    if isinstance(taxonomy_json, str):
        taxonomy_json = json.loads(taxonomy_json)  # Parse JSON string if needed

    # Build tree structure
    tree = defaultdict(list)
    for feature, (_, parent) in taxonomy_json.items():
        tree[parent].append(feature)

    def display_tree(parent="Root", level=0):
        """Recursively formats the tree structure with icons and connectors."""
        if parent not in tree:
            return ""

        result = ""
        for idx, child in enumerate(tree[parent]):
            is_parent = child in tree  # Check if child has further children
            icon = "📂" if is_parent else "📄"  # Folder for parents, file for leaf nodes
            connector = "└── " if idx == len(tree[parent]) - 1 else "├── "  # Tree connectors
            result += "  " * level + connector + f"{icon} {child}\n"
            result += display_tree(child, level + 1)  # Recursive call for children

        return result

    return f"\n📌 **Taxonomy Structure:**\n\n{display_tree('Root')}".strip()



def format_taxonomy_deepseek(taxonomy: Union[str, dict]) -> str:
    """
    Accept either a dict loaded from JSON (key -> [level, category])
    or a filepath to a JSON file. Returns a formatted string and prints it.
    """
    # load if a path was passed
    if isinstance(taxonomy, str):
        with open(taxonomy, "r", encoding="utf-8") as f:
            taxonomy = json.load(f)

    # validate structure
    if not isinstance(taxonomy, dict):
        raise ValueError("taxonomy must be a dict or a path to a JSON file")

    # build mapping: category -> list of (feature, level)
    cat_map = defaultdict(list)
    for feat, meta in taxonomy.items():
        # meta expected like [level, category] but be tolerant
        if isinstance(meta, (list, tuple)) and len(meta) >= 2:
            level, category = meta[0], meta[1]
        elif isinstance(meta, dict):
            level = meta.get("level", None)
            category = meta.get("category", "Uncategorized")
        else:
            level = None
            category = "Uncategorized"
        cat_map[str(category)].append((feat, level))

    # create printable output sorted by category then feature
    lines = []
    for category in sorted(cat_map.keys()):
        lines.append(f"Category: {category}")
        # sort features by level (if numeric) then name
        items = sorted(cat_map[category], key=lambda x: (x[1] if isinstance(x[1], (int,float)) else 999, x[0]))
        for feat, level in items:
            lines.append(f"  - {feat} (level={level})")
        lines.append("")  # blank line

    out = "\n".join(lines)
    print(out)   # ensure output appears in notebooks/console
    return out


In [14]:
df.columns

Index(['datetime', 'machineID', 'volt', 'rotate', 'pressure', 'vibration',
       'model', 'age', 'errorID', 'comp', 'failure', 'target', 'volt_lag_1h',
       'volt_lag_6h', 'volt_lag_12h', 'volt_lag_24h', 'volt_mean_24h',
       'volt_std_24h', 'volt_min_24h', 'volt_max_24h', 'volt_mean_6h',
       'volt_std_6h', 'rotate_lag_1h', 'rotate_lag_6h', 'rotate_lag_12h',
       'rotate_lag_24h', 'rotate_mean_24h', 'rotate_std_24h', 'rotate_min_24h',
       'rotate_max_24h', 'rotate_mean_6h', 'rotate_std_6h', 'pressure_lag_1h',
       'pressure_lag_6h', 'pressure_lag_12h', 'pressure_lag_24h',
       'pressure_mean_24h', 'pressure_std_24h', 'pressure_min_24h',
       'pressure_max_24h', 'pressure_mean_6h', 'pressure_std_6h',
       'vibration_lag_1h', 'vibration_lag_6h', 'vibration_lag_12h',
       'vibration_lag_24h', 'vibration_mean_24h', 'vibration_std_24h',
       'vibration_min_24h', 'vibration_max_24h', 'vibration_mean_6h',
       'vibration_std_6h', 'errorID_lag_1h', 'errorID_lag_6h',


In [18]:
tx_index = 2
taxonomy_model = [gpt4o_taxonomy, llama3_taxonomy, gemma2_taxonomy, gpt4o_edited_taxonomy]

print(format_taxonomy(taxonomy_model[tx_index]))

# print("\n\n\n\n\n")

# For deepSeek run the following method
# print(format_taxonomy_deepseek(deepseek_taxonomy))

📌 **Taxonomy Structure:**

├── 📄 Type
├── 📂 Timestamp
  ├── 📄 datetime
  ├── 📄 hour
  ├── 📄 day_of_week
  ├── 📄 is_weekend
  └── 📄 is_working_hours
├── 📂 Machine
  ├── 📄 machineID
  ├── 📄 model
  └── 📄 age
├── 📂 Sensor
  ├── 📂 volt
    ├── 📄 volt_lag_1h
    ├── 📄 volt_lag_6h
    ├── 📄 volt_lag_12h
    ├── 📄 volt_lag_24h
    ├── 📄 volt_mean_24h
    ├── 📄 volt_std_24h
    ├── 📄 volt_min_24h
    ├── 📄 volt_max_24h
    ├── 📄 volt_mean_6h
    └── 📄 volt_std_6h
  ├── 📂 rotate
    ├── 📄 rotate_lag_1h
    ├── 📄 rotate_lag_6h
    ├── 📄 rotate_lag_12h
    ├── 📄 rotate_lag_24h
    ├── 📄 rotate_mean_24h
    ├── 📄 rotate_std_24h
    ├── 📄 rotate_min_24h
    ├── 📄 rotate_max_24h
    ├── 📄 rotate_mean_6h
    └── 📄 rotate_std_6h
  ├── 📂 pressure
    ├── 📄 pressure_lag_1h
    ├── 📄 pressure_lag_6h
    ├── 📄 pressure_lag_12h
    ├── 📄 pressure_lag_24h
    ├── 📄 pressure_mean_24h
    ├── 📄 pressure_std_24h
    ├── 📄 pressure_min_24h
    ├── 📄 pressure_max_24h
    ├── 📄 pressure_mean_6h
    └── 📄 pressure

In [ ]:
shap_features = {
    'feature': df_shap["feature_name"].to_list()
}
features_to_find = shap_features["feature"]

taxonomy_model = [gpt4o_taxonomy, llama3_taxonomy, gemma2_taxonomy, deepseek_taxonomy, gpt4o_edited_taxonomy, ]
index = 4

match index:
    case 0:
        print("GPT-4o")
    case 1:
        print("Llama 3.3")
    case 2:
        print("Gemma2")
    case 3:
        print("DeepSeek")
    case 4:
        print("GPT-4o Edited")
    case _:
        print("Unknown model")

try:
    # Get paths for all selected features
    feature_paths = get_taxonomy_paths(taxonomy_model[index], features_to_find)

    # Print the results in a formatted way, removing "Sensor_Reading --> " from the start
    print("\n=== Taxonomy Paths ===\n")
    for feature, path in feature_paths.items():
        if path.startswith("Sensor_Readings --> "):
            path = path[len("Sensor_Readings --> "):]
        print(f"{path}")
except IndexError as e:
    print("Taxonomy model index is out of reach")

<br> <br>

#### See the hierarchy

In [ ]:
# Test the function with the openai_taxonomy data

shap_features = {
    'feature': df_shap["feature_name"].to_list()
}
features_to_find = shap_features["feature"]

taxonomy_model = [gpt4o_taxonomy, llama3_taxonomy, gemma2_taxonomy, deepseek_taxonomy, gpt4o_edited_taxonomy, ]
index = 4

match index:
    case 0:
        print("GPT-4o")
    case 1:
        print("Llama 3.3")
    case 2:
        print("Gemma2")
    case 3:
        print("DeepSeek")
    case 4:
        print("GPT-4o Edited")
    case _:
        print("Unknown model")

try:
    # Get paths for all selected features
    feature_paths = get_taxonomy_paths(taxonomy_model[index], features_to_find)

    # Print the results in a formatted way, removing "Sensor_Reading --> " from the start
    print("\n=== Taxonomy Paths ===\n")
    for feature, path in feature_paths.items():
        if path.startswith("Sensor_Readings --> "):
            path = path[len("Sensor_Readings --> "):]
        print(f"{path}")
except IndexError as e:
    print("Taxonomy model index is out of reach")

<br> <br>

---

<br> <br> <br>

## Visualize The subgraph - Advanced Visualization - Target included


In [ ]:
def create_interactive_visualization_with_target(df_shap, structured_adjacency, top_n_sub_nodes=None, filename="interactive_causal_graph.html", width="100%", height="1000px"):
    """
    Creates an interactive HTML visualization of the causal graph containing top SHAP features.
    
    This version includes the top_n_sub_nodes filtering logic and creates an interactive visualization
    using pyvis instead of matplotlib.

    Args:
        df_shap (pd.DataFrame): DataFrame with a 'feature_name' column identifying the core features.
        structured_adjacency (list): List of dicts with 'source_feature', 'target_feature', 
                                     and 'effect_strength' keys.
        top_n_sub_nodes (int, optional): The number of top related nodes to include in the graph,
                                         ranked by absolute effect_strength. If None, all directly
                                         connected nodes are included. Defaults to None.
        filename (str): The name of the output HTML file. Defaults to "interactive_causal_graph.html".
        width (str): Width of the visualization. Defaults to "100%".
        height (str): Height of the visualization. Defaults to "1000px".
    
    Returns:
        str: JSON string of the subgraph (same as original function).
    """
    import networkx as nx
    import json
    import pandas as pd
    from pyvis.network import Network

    top_features = set(df_shap["feature_name"])
    target_variable = 'target'  # Define the target variable name
    
    # Store original effect strengths for JSON output
    edge_strengths = {}
    
    # If top_n_sub_nodes is specified, filter to the most influential relationships
    if top_n_sub_nodes is not None and isinstance(top_n_sub_nodes, int) and top_n_sub_nodes >= 0:
        # 1. Find all edges connecting top_features to other nodes (sub-nodes), excluding target
        relevant_edges = []
        for edge in structured_adjacency:
            source, target = edge["source_feature"], edge["target_feature"]
            strength = abs(edge.get("effect_strength", 0))  # Use absolute strength for ranking
            
            # Find relationships where one node is in top_features and the other is not (excluding target variable)
            if source in top_features and target not in top_features and target != target_variable:
                relevant_edges.append({
                    'top_feature': source,
                    'sub_node': target,
                    'strength': strength,
                    'original_edge': edge
                })
            elif target in top_features and source not in top_features and source != target_variable:
                relevant_edges.append({
                    'top_feature': target,
                    'sub_node': source,
                    'strength': strength,
                    'original_edge': edge
                })
        
        # 2. Sort all relationships by effect strength and select top N
        sorted_relationships = sorted(relevant_edges, key=lambda x: x['strength'], reverse=True)
        top_relationships = sorted_relationships[:top_n_sub_nodes]
        
        # 3. Get the set of sub-nodes from top N relationships
        top_sub_nodes_set = {rel['sub_node'] for rel in top_relationships}
        
        print(f"Selected top {len(top_relationships)} relationships with sub-nodes: {top_sub_nodes_set}")
        for i, rel in enumerate(top_relationships, 1):
            print(f"  {i}. {rel['top_feature']} ↔ {rel['sub_node']} (strength: {rel['strength']:.4f})")
        
        # 4. The final set of nodes includes top features, selected sub-nodes, AND target variable
        allowed_nodes = top_features.union(top_sub_nodes_set).union({target_variable})
        
        # 5. Filter edges where both source and target are in our allowed set
        sub_edges = []
        edge_labels = {}
        for edge in structured_adjacency:
            source, target = edge["source_feature"], edge["target_feature"]
            if source in allowed_nodes and target in allowed_nodes:
                sub_edges.append((source, target))
                effect_strength = edge.get("effect_strength", 0)
                edge_labels[(source, target)] = f'{effect_strength:.3f}'
                edge_strengths[(source, target)] = effect_strength  # Store original value
    else:
        # Original logic: include all nodes directly connected to any top_feature, PLUS target variable
        sub_edges = []
        edge_labels = {}
        allowed_nodes = set()
        
        for edge in structured_adjacency:
            source, target = edge["source_feature"], edge["target_feature"]
            # Include edges connected to top_features OR target variable
            if (source in top_features) or (target in top_features) or (source == target_variable) or (target == target_variable):
                sub_edges.append((source, target))
                effect_strength = edge.get("effect_strength", 0)
                edge_labels[(source, target)] = f'{effect_strength:.3f}'
                edge_strengths[(source, target)] = effect_strength  # Store original value
                allowed_nodes.add(source)
                allowed_nodes.add(target)

    # Create NetworkX graph
    G = nx.DiGraph()
    G.add_edges_from(sub_edges)
    
    # Add edge labels as attributes to the graph
    for (u, v), label in edge_labels.items():
        if G.has_edge(u, v):
            G.edges[u, v]['label'] = label

    # If the graph is empty, return an empty JSON and skip visualization
    if not G.nodes():
        print("Warning: The resulting graph is empty. No relationships found based on the criteria.")
        return json.dumps({"nodes": [], "edges": []}, indent=2)

    # Create interactive visualization using pyvis
    print(f"Creating interactive visualization with {len(G.nodes())} nodes and {len(G.edges())} edges...")
    
    net = Network(height=height, width=width, notebook=False, directed=True, cdn_resources='in_line')

    # Set physics options for better layout and interactivity
    net.set_options("""
    var options = {
      "physics": {
        "enabled": true,
        "repulsion": { 
          "centralGravity": 0.2, 
          "springLength": 200, 
          "nodeDistance": 300,
          "damping": 0.1
        },
        "minVelocity": 0.75,
        "solver": "repulsion",
        "timestep": 0.22,
        "stabilization": {"iterations": 150}
      },
      "edges": {
        "arrows": {
          "to": {"enabled": true, "scaleFactor": 1.5}
        },
        "smooth": {
          "enabled": true,
          "type": "continuous"
        }
      },
      "interaction": {
        "dragNodes": true,
        "dragView": true,
        "zoomView": true,
        "selectConnectedEdges": true,
        "hover": true,
        "multiselect": true,
        "keyboard": {
          "enabled": true
        }
      },
      "manipulation": {
        "enabled": false
      }
    }
    """)

    # Add nodes to the pyvis network with enhanced interactivity
    for node in G.nodes():
        is_top = node in top_features
        is_target = node == target_variable  
        
        # Determine node styling based on type
        if is_target:
            color = 'red'
            node_type = '🎯 TARGET VARIABLE'
            size = 35  # Larger size for target
        elif is_top:
            color = 'orange'
            node_type = '⭐ Top SHAP Feature'
            size = 25
        else:
            color = 'skyblue'
            node_type = '🔗 Related Feature'
            size = 20
            
        net.add_node(
            node, 
            label=node, 
            color=color,
            shape='box', 
            shadow=True, 
            font={'size': 18, 'color': 'black', 'strokeWidth': 2, 'strokeColor': 'white'},
            size=size,
            borderWidth=3 if is_target else 2,
            borderWidthSelected=5 if is_target else 4,
            title=f"{node_type}: {node}<br/>Click and drag to move!"  # Enhanced tooltip
        )

    # Add edges to the pyvis network with enhanced styling
    for u, v in G.edges():
        label = G.edges[u, v].get('label', '')
        is_highlight = (u in top_features and v in top_features)
        is_target_edge = (u == target_variable or v == target_variable)
        
        # Special styling for edges connected to target
        if is_target_edge:
            edge_color = 'darkred'
            edge_width = 5
            edge_title = f"🎯 TARGET CONNECTION - Effect strength: {label}<br/>From: {u} → To: {v}"
        elif is_highlight:
            edge_color = 'red'
            edge_width = 4
            edge_title = f"📊 SHAP-SHAP Connection - Effect strength: {label}<br/>From: {u} → To: {v}"
        else:
            edge_color = 'gray'
            edge_width = 2
            edge_title = f"📊 Effect strength: {label}<br/>From: {u} → To: {v}"
            
        net.add_edge(
            u, v, 
            label=label, 
            color=edge_color,
            width=edge_width,
            title=edge_title,  # Enhanced tooltip
            font={'size': 12, 'color': 'blue', 'strokeWidth': 1, 'strokeColor': 'white'}
        )

    # Generate and save HTML content with proper encoding
    html_content = net.generate_html()
    try:
        with open(filename, "w", encoding="utf-8") as f:
            f.write(html_content)
        print(f"[SUCCESS] Interactive graph saved to '{filename}'. Open this file in your browser.")
    except Exception as e:
        print(f"[ERROR] Could not save the HTML file. Reason: {e}")
        return json.dumps({"nodes": [], "edges": []}, indent=2)

    # Generate JSON output (same as original function, but include target info)
    subgraph_json = {
        "nodes": [
            {
                "id": node, 
                "highlight": node in top_features,
                "is_target": node == target_variable,
                "node_type": "target" if node == target_variable else ("shap_feature" if node in top_features else "related_feature")
            }
            for node in G.nodes()
        ],
        "edges": [
            {
                "source": u,
                "target": v,
                "highlight": (u in top_features and v in top_features),
                "is_target_connection": (u == target_variable or v == target_variable),
                "effect_strength": edge_strengths.get((u, v), 0.0)  # Use original numeric value
            }
            for u, v in G.edges()
        ]
    }
    
    return json.dumps(subgraph_json, indent=2)

In [ ]:
# Example usage with custom size:
# For a larger plot:
# result = create_interactive_visualization_with_target(df_shap, structured_adjacency, top_n_sub_nodes=7, filename="large_graph.html", width="1400px", height="1200px")
# 
# For full screen:
result = create_interactive_visualization_with_target(df_shap, structured_adjacency, top_n_sub_nodes=None, filename="interactive_causal_graph_with_target-21-08-2025.html", width="100vw" , height="100vh")

In [ ]:
df_shap

<br> <br> <br>

---

<br> <br> <br>

## Counterfactual Analysis using Dice

[Dice-ML](https://pypi.org/project/dice-ml/#:~:text=With%20DiCE%2C%20generating%20explanations%20is%20a%20simple%20three-step,models%2C%20with%20or%20without%20their%20original%20training%20data.)

In [ ]:
# random_row

In [ ]:
# def generate_dice_counterfactuals_for_important_features_robust(model, random_row: pd.DataFrame, important_features: list, X_train: pd.DataFrame, y_train: pd.Series, y_test: pd.Series, 
#                                                          num_counterfactuals: int = 5, desired_class: int = 0, random_seed: int = 42 ) -> pd.DataFrame:
#     """
#     Robust version of DiCE counterfactual generation with fallback strategies.
    
#     Args:
#         model: Trained ML model
#         random_row: Single row DataFrame from X_test to generate counterfactuals for
#         important_features: List of important feature names from SHAP
#         X_train: Training data for DiCE
#         y_train: Training labels for DiCE
#         y_test: Test labels to get the actual target for random_row
#         num_counterfactuals: Number of counterfactuals to generate
#         desired_class: Target class (0 for no failure, 1 for failure)
#         random_seed: Random seed for reproducibility
        
#     Returns:
#         DataFrame with counterfactuals showing only changed features
#     """
#     print("🎯 Generating DiCE Counterfactuals for Important Features (Robust)")
#     print("=" * 60)
    
#     # Get the actual target value for the random_row - Fix for numpy array
#     row_index = int(random_row.index.values[0])
#     if hasattr(y_test, 'loc'):
#         actual_target = y_test.loc[row_index]
#     else:
#         # Convert numpy array index to position in X_test
#         X_test_index = random_row.index[0]
#         position = list(X_test.index).index(X_test_index) if hasattr(X_test, 'index') else row_index
#         actual_target = y_test[position]
    
#     print(f"📊 Original row index: {random_row.index.values[0]}")
#     print(f"🎯 Actual target value: {actual_target}")
#     print(f"🔄 Desired target value: {desired_class}")
    
#     # Prepare training data with target for DiCE
#     train_data = X_train.copy()
#     train_data['target'] = y_train
    
#     # Prepare data for DiCE
#     d = dice_ml.Data(
#         dataframe=train_data,
#         continuous_features=list(X_train.columns),
#         outcome_name='target'
#     )
    
#     # Create DiCE model
#     m = dice_ml.Model(model=model, backend="sklearn")
    
#     # Try multiple strategies with different configurations
#     strategies = [
#         # Strategy 1: Original approach with important features only
#         {
#             'features_to_vary': important_features,
#             'method': 'random',
#             'description': 'Important features only (random method)'
#         },
#         # Strategy 2: Reduce number of features to vary
#         {
#             'features_to_vary': important_features[:5],  # Top 5 features
#             'method': 'random',
#             'description': 'Top 5 important features (random method)'
#         },
#         # Strategy 3: Allow all features to vary
#         {
#             'features_to_vary': list(X_train.columns),
#             'method': 'random',
#             'description': 'All features (random method)'
#         },
#         # Strategy 4: Use genetic algorithm method with important features
#         {
#             'features_to_vary': important_features,
#             'method': 'genetic',
#             'description': 'Important features (genetic method)'
#         },
#         # Strategy 5: Use KD-Tree method with all features
#         {
#             'features_to_vary': list(X_train.columns),
#             'method': 'kdtree',
#             'description': 'All features (kdtree method)'
#         }
#     ]
    
#     for i, strategy in enumerate(strategies, 1):
#         print(f"\n🔄 Strategy {i}: {strategy['description']}")
#         print(f"   Features to vary: {len(strategy['features_to_vary'])}")
        
#         try:
#             # Create DiCE explainer with current strategy
#             exp = dice_ml.Dice(d, m, method=strategy['method'])
            
#             # Generate counterfactuals
#             query_for_dice = random_row.copy()
            
#             dice_exp = exp.generate_counterfactuals(
#                 query_for_dice,
#                 total_CFs=num_counterfactuals,
#                 desired_class=desired_class,
#                 features_to_vary=strategy['features_to_vary'],
#                 random_seed=random_seed
#             )
            
#             # Extract counterfactuals
#             counterfactuals_df = dice_exp.cf_examples_list[0].final_cfs_df
            
#             if not counterfactuals_df.empty:
#                 print(f"✅ SUCCESS with Strategy {i}! Generated {len(counterfactuals_df)} counterfactuals")
                
#                 # Analyze changes - use the original important_features for consistency
#                 return analyze_counterfactual_changes(
#                     counterfactuals_df, random_row, model, important_features
#                 )
#             else:
#                 print(f"❌ Strategy {i} failed: No counterfactuals generated")
                
#         except Exception as e:
#             print(f"❌ Strategy {i} failed with error: {str(e)}")
#             continue
    
#     # If all strategies fail, try a simplified approach
#     print("\n🔄 Final fallback: Simplified counterfactual generation")
#     try:
#         return generate_simple_counterfactuals(random_row, model, important_features, num_counterfactuals)
#     except Exception as e:
#         print(f"❌ All strategies failed. Final error: {str(e)}")
#         return pd.DataFrame()



# def analyze_counterfactual_changes(counterfactuals_df, random_row, model, important_features):
#     """
#     Analyze changes between original row and counterfactuals.
#     """
#     # Verify predictions changed
#     original_prediction = model.predict(random_row)[0]
#     original_proba = model.predict_proba(random_row)[0]
    
#     print(f"📊 Original prediction: {original_prediction}")
#     print(f"📊 Original probabilities: {original_proba}")
    
#     # Analyze changes - compare with original row
#     original_values = random_row.iloc[0]
#     changes_data = []
    
#     for idx, cf_row in counterfactuals_df.iterrows():
#         # Get counterfactual features (remove target column if present)
#         cf_features = cf_row.drop('target') if 'target' in cf_row.index else cf_row
#         cf_features_df = cf_features.to_frame().T
        
#         # Verify prediction changed
#         cf_prediction = model.predict(cf_features_df)[0]
#         cf_proba = model.predict_proba(cf_features_df)[0]
        
#         change_info = {
#             'counterfactual_id': idx,
#             'original_prediction': original_prediction,
#             'cf_prediction': cf_prediction,
#             'prediction_changed': cf_prediction != original_prediction,
#             'original_proba_class_1': original_proba[1] if len(original_proba) > 1 else original_proba[0],
#             'cf_proba_class_1': cf_proba[1] if len(cf_proba) > 1 else cf_proba[0]
#         }
        
#         # Check which features changed
#         changed_features = []
#         for feature in important_features:
#             if feature in original_values.index and feature in cf_features.index:
#                 original_val = original_values[feature]
#                 cf_val = cf_features[feature]
                
#                 if abs(original_val - cf_val) > 1e-6:  # Feature changed
#                     changed_features.append(feature)
#                     change_info[f'{feature}_original'] = original_val
#                     change_info[f'{feature}_counterfactual'] = cf_val
#                     change_info[f'{feature}_change'] = cf_val - original_val
#                     change_info[f'{feature}_percent_change'] = ((cf_val - original_val) / original_val * 100) if original_val != 0 else float('inf')
        
#         change_info['changed_features'] = ', '.join(changed_features)
#         change_info['num_changed_features'] = len(changed_features)
#         changes_data.append(change_info)
    
#     # Create summary DataFrame
#     changes_df = pd.DataFrame(changes_data)
    
#     # Display summary
#     print(f"\n📊 COUNTERFACTUAL ANALYSIS SUMMARY:")
#     print(f"   Original prediction: {original_prediction} (class 1 prob: {change_info['original_proba_class_1']:.3f})")
#     print(f"   Generated counterfactuals: {len(counterfactuals_df)}")
#     print(f"   Successful prediction changes: {sum(changes_df['prediction_changed'])}")
    
#     return changes_df



# def generate_simple_counterfactuals(random_row, model, important_features, num_counterfactuals):
#     """
#     Simple fallback counterfactual generation using random perturbations.
#     """
#     print("🔄 Generating simple counterfactuals using random perturbations...")
    
#     original_prediction = model.predict(random_row)[0]
#     original_proba = model.predict_proba(random_row)[0]
    
#     changes_data = []
#     successful_cfs = 0
    
#     for i in range(num_counterfactuals * 10):  # Try more attempts
#         # Create perturbed version
#         perturbed_row = random_row.copy()
        
#         # Randomly perturb important features
#         for feature in important_features[:5]:  # Use top 5 features
#             if feature in perturbed_row.columns:
#                 original_val = perturbed_row[feature].iloc[0]
#                 # Add random noise (±20% of original value)
#                 noise = np.random.normal(0, abs(original_val) * 0.2)
#                 perturbed_row[feature] = original_val + noise
        
#         # Check if prediction changed
#         try:
#             cf_prediction = model.predict(perturbed_row)[0]
#             cf_proba = model.predict_proba(perturbed_row)[0]
            
#             if cf_prediction != original_prediction:
#                 change_info = {
#                     'counterfactual_id': successful_cfs,
#                     'original_prediction': original_prediction,
#                     'cf_prediction': cf_prediction,
#                     'prediction_changed': True,
#                     'original_proba_class_1': original_proba[1] if len(original_proba) > 1 else original_proba[0],
#                     'cf_proba_class_1': cf_proba[1] if len(cf_proba) > 1 else cf_proba[0],
#                     'changed_features': ', '.join(important_features[:5]),
#                     'num_changed_features': 5
#                 }
#                 changes_data.append(change_info)
#                 successful_cfs += 1
                
#                 if successful_cfs >= num_counterfactuals:
#                     break
                    
#         except Exception:
#             continue
    
#     if changes_data:
#         print(f"✅ Generated {len(changes_data)} simple counterfactuals")
#         return pd.DataFrame(changes_data)
#     else:
#         print("❌ Failed to generate any counterfactuals")
#         return pd.DataFrame()


# def feature_display_label(feature):
#     """
#     Convert feature names to friendly display labels.
#     """
#     # Simple mapping - you can customize this based on your features
#     label_mapping = {
#         'volt': 'Voltage',
#         'rotate': 'Rotation',
#         'pressure': 'Pressure',
#         'vibration': 'Vibration',
#         'model': 'Model',
#         'age': 'Age',
#         'target': 'Target',
#         'errorID': 'Error ID',
#         'comp': 'Component'
#     }
    
#     return label_mapping.get(feature, feature.replace('_', ' ').title())


# def create_llm_ready_counterfactual_summary(counterfactual_results: pd.DataFrame, model, random_row: pd.DataFrame, important_features: list) -> pd.DataFrame:
#     """
#     Create a clean summary DataFrame with essential counterfactual information for LLM explanation.
#     Outputs include:
#       - 'changed_feature_names' (original feature keys, comma separated)
#       - 'changed_feature_display' (Friendly Name (original) comma separated)  <-- new
#       - 'feature_changes_summary' using friendly display labels
#     """
#     print("📊 Creating LLM-Ready Counterfactual Summary")
#     print("=" * 50)

#     if counterfactual_results.empty:
#         print("⚠️ No counterfactual results to summarize")
#         return pd.DataFrame()

#     # Filter to only successful counterfactuals
#     successful_cfs = counterfactual_results[counterfactual_results['prediction_changed'] == True]

#     if successful_cfs.empty:
#         print("⚠️ No successful counterfactuals found")
#         return pd.DataFrame()

#     # Get original prediction info
#     original_prediction = model.predict(random_row)[0]
#     original_proba = model.predict_proba(random_row)[0]

#     llm_ready_data = []

#     for _, row in successful_cfs.iterrows():
#         changed_features = []
#         changed_displays = []
#         feature_changes_text = []

#         for feature in important_features:
#             key_change = f'{feature}_change'
#             if key_change in row and pd.notna(row[key_change]):
#                 original_val = row.get(f'{feature}_original', None)
#                 new_val = row.get(f'{feature}_counterfactual', None)
#                 change_val = row.get(key_change, None)
#                 percent_change = row.get(f'{feature}_percent_change', None)

#                 changed_features.append(feature)
#                 changed_displays.append(feature_display_label(feature))

#                 # Use friendly display in the detail text: Friendly (orig): old -> new (Delta..., %)
#                 try:
#                     orig_fmt = f"{original_val:.3f}" if pd.notna(original_val) else str(original_val)
#                     new_fmt = f"{new_val:.3f}" if pd.notna(new_val) else str(new_val)
#                     change_fmt = f"{change_val:+.3f}" if pd.notna(change_val) else str(change_val)
#                     pct_fmt = f"{percent_change:+.1f}%" if pd.notna(percent_change) and np.isfinite(percent_change) else ("inf%" if percent_change == float('inf') else str(percent_change))
#                 except Exception:
#                     orig_fmt, new_fmt, change_fmt, pct_fmt = original_val, new_val, change_val, percent_change

#                 feature_changes_text.append(
#                     f"{feature_display_label(feature)}: {orig_fmt} -> {new_fmt} (Delta{change_fmt}, {pct_fmt})"
#                 )

#         llm_data = {
#             'counterfactual_id': row['counterfactual_id'],
#             'changed_feature_names': ', '.join(changed_features),
#             'changed_feature_display': ', '.join(changed_displays),
#             'num_changed_features': len(changed_features),
#             'original_prediction': original_prediction,
#             'original_probability_failure': round(original_proba[1], 3),
#             'counterfactual_prediction': row['cf_prediction'],
#             'counterfactual_probability_failure': round(row['cf_proba_class_1'], 3),
#             'prediction_flip': f"{original_prediction} to {row['cf_prediction']}",
#             'probability_change': round(row['cf_proba_class_1'] - original_proba[1], 3),
#             'feature_changes_summary': '; '.join(feature_changes_text)
#         }

#         llm_ready_data.append(llm_data)

#     llm_df = pd.DataFrame(llm_ready_data)
#     print(f"✅ Created LLM-ready summary with {len(llm_df)} successful counterfactuals")
#     return llm_df


# # Use the robust version instead
# changes_df = generate_dice_counterfactuals_for_important_features_robust(
#     model=model,
#     random_row=random_row,
#     important_features=df_shap["feature_name"].tolist(),
#     X_train=X_train,
#     y_train=y_train,
#     y_test=y_test,
#     num_counterfactuals=10,
#     desired_class=0,
#     random_seed=42
# )

# # Only run LLM summary if we have results
# if not changes_df.empty:
#     llm_df = create_llm_ready_counterfactual_summary(
#         counterfactual_results=changes_df,
#         model=model,
#         random_row=random_row,
#         important_features=df_shap["feature_name"].tolist()
#     )
#     display(llm_df)
# else:
#     print("⚠️ No counterfactuals generated. Consider using different parameters or methods.")


In [ ]:
# llm_df = create_llm_ready_counterfactual_summary(
#     counterfactual_results=changes_df,
#     model=model,
#     random_row=random_row,
#     important_features=df_shap["feature_name"].tolist()  # Use the correct variable
# )



<br> <br> <br>

---

## Perturb Multiple Features 

In [ ]:
# import numpy as np
# import pandas as pd
# from itertools import combinations
# import random

# def generate_prediction_changing_counterfactuals(model, random_row: pd.DataFrame, user_features: list, 
#                                                num_counterfactuals: int = 5, max_iterations: int = 1000,
#                                                perturbation_strength: float = 0.1, random_seed: int = 42) -> pd.DataFrame:
#     """
#     Generate counterfactuals by systematically perturbing features until prediction changes.
#     Focus on finding minimal feature changes that cause prediction flip.
    
#     Args:
#         model: Trained ML model
#         random_row: Single row DataFrame to generate counterfactuals for
#         user_features: List of features to perturb (in order of priority)
#         num_counterfactuals: Number of successful counterfactuals to find
#         max_iterations: Maximum attempts per counterfactual
#         perturbation_strength: Initial strength of perturbation (as fraction of original value)
#         random_seed: Random seed for reproducibility
        
#     Returns:
#         DataFrame with successful counterfactuals showing which features caused prediction change
#     """
#     print("🎯 Finding Prediction-Changing Counterfactuals")
#     print("=" * 60)
    
#     np.random.seed(random_seed)
#     random.seed(random_seed)
    
#     # Validate user features
#     available_features = set(random_row.columns)
#     valid_user_features = [f for f in user_features if f in available_features]
#     invalid_features = [f for f in user_features if f not in available_features]
    
#     if invalid_features:
#         print(f"⚠️ Warning: These features are not available: {invalid_features}")
    
#     if not valid_user_features:
#         print("❌ Error: No valid features provided.")
#         return pd.DataFrame()
    
#     print(f"✅ Will perturb features in this order: {valid_user_features}")
    
#     # Get original prediction
#     original_prediction = model.predict(random_row)[0]
#     original_proba = model.predict_proba(random_row)[0]
#     original_values = random_row.iloc[0].copy()
    
#     print(f"📊 Original prediction: {original_prediction}")
#     print(f"📊 Original probability: {original_proba}")
#     print(f"🎯 Looking for prediction flip to: {1 - original_prediction}")
    
#     successful_counterfactuals = []
    
#     # Strategy 1: Minimal Feature Changes (start with 1 feature, then 2, etc.)
#     print(f"\n🔍 Strategy 1: Finding minimal feature changes...")
#     for num_features_to_change in range(1, min(len(valid_user_features) + 1, 6)):  # Try up to 5 features
#         print(f"\n   Trying {num_features_to_change} feature(s) at a time...")
        
#         # Try all combinations of features of this size
#         for feature_combination in combinations(valid_user_features, num_features_to_change):
#             if len(successful_counterfactuals) >= num_counterfactuals:
#                 break
                
#             counterfactual = find_counterfactual_for_features(
#                 model, random_row, list(feature_combination), original_prediction,
#                 max_iterations, perturbation_strength, len(successful_counterfactuals)
#             )
            
#             if counterfactual is not None:
#                 successful_counterfactuals.append(counterfactual)
#                 print(f"   ✅ Found counterfactual {len(successful_counterfactuals)} using: {list(feature_combination)}")
        
#         if len(successful_counterfactuals) >= num_counterfactuals:
#             break
    
#     # Strategy 2: Systematic single-feature perturbation if we need more
#     if len(successful_counterfactuals) < num_counterfactuals:
#         print(f"\n🔍 Strategy 2: Systematic single-feature exploration...")
#         for feature in valid_user_features:
#             if len(successful_counterfactuals) >= num_counterfactuals:
#                 break
                
#             counterfactual = find_counterfactual_single_feature_systematic(
#                 model, random_row, feature, original_prediction, len(successful_counterfactuals)
#             )
            
#             if counterfactual is not None:
#                 successful_counterfactuals.append(counterfactual)
#                 print(f"   ✅ Found counterfactual {len(successful_counterfactuals)} using: {feature}")
    
#     # Strategy 3: Aggressive multi-feature perturbation if still need more
#     if len(successful_counterfactuals) < num_counterfactuals:
#         print(f"\n🔍 Strategy 3: Aggressive multi-feature perturbation...")
#         remaining_needed = num_counterfactuals - len(successful_counterfactuals)
        
#         for i in range(remaining_needed * 3):  # Try multiple times
#             if len(successful_counterfactuals) >= num_counterfactuals:
#                 break
                
#             counterfactual = find_counterfactual_aggressive(
#                 model, random_row, valid_user_features, original_prediction, 
#                 len(successful_counterfactuals), max_iterations
#             )
            
#             if counterfactual is not None:
#                 successful_counterfactuals.append(counterfactual)
#                 print(f"   ✅ Found counterfactual {len(successful_counterfactuals)} (aggressive)")
    
#     # Convert to DataFrame
#     if successful_counterfactuals:
#         results_df = pd.DataFrame(successful_counterfactuals)
#         print(f"\n🎉 SUCCESS: Generated {len(results_df)} counterfactuals that changed the prediction!")
        
#         # Add summary statistics
#         print(f"\n📊 SUMMARY:")
#         print(f"   Original prediction: {original_prediction}")
#         print(f"   Successful prediction flips: {len(results_df)}")
        
#         # Show which features were most effective
#         all_changed_features = []
#         for cf in successful_counterfactuals:
#             all_changed_features.extend(cf['changed_features_list'])
        
#         from collections import Counter
#         feature_effectiveness = Counter(all_changed_features)
#         print(f"   Most effective features: {dict(feature_effectiveness.most_common(5))}")
        
#         return results_df
#     else:
#         print(f"\n❌ Could not find any counterfactuals that change the prediction.")
#         print(f"   Try increasing perturbation_strength or max_iterations.")
#         return pd.DataFrame()


# def find_counterfactual_for_features(model, random_row, features_to_perturb, original_prediction, 
#                                    max_iterations, perturbation_strength, cf_id):
#     """Find counterfactual by perturbing specific features until prediction changes."""
    
#     original_values = random_row.iloc[0].copy()
    
#     for iteration in range(max_iterations):
#         # Create perturbed version
#         perturbed_row = random_row.copy()
#         perturbations = {}
        
#         for feature in features_to_perturb:
#             if feature in perturbed_row.columns:
#                 original_val = original_values[feature]
                
#                 # Use increasing perturbation strength over iterations
#                 current_strength = perturbation_strength * (1 + iteration * 0.1)
                
#                 # Try different perturbation strategies
#                 if iteration % 4 == 0:
#                     # Strategy A: Random normal perturbation
#                     noise = np.random.normal(0, abs(original_val) * current_strength)
#                 elif iteration % 4 == 1:
#                     # Strategy B: Fixed positive perturbation
#                     noise = abs(original_val) * current_strength * np.random.choice([1, -1])
#                 elif iteration % 4 == 2:
#                     # Strategy C: Large step perturbation
#                     noise = original_val * current_strength * np.random.choice([2, -2])
#                 else:
#                     # Strategy D: Exponential perturbation
#                     noise = original_val * (np.exp(current_strength) - 1) * np.random.choice([1, -1])
                
#                 new_val = original_val + noise
#                 perturbed_row.loc[perturbed_row.index[0], feature] = new_val
#                 perturbations[feature] = {
#                     'original': original_val,
#                     'new': new_val,
#                     'change': noise,
#                     'percent_change': (noise / original_val * 100) if original_val != 0 else float('inf')
#                 }
        
#         # Check if prediction changed
#         try:
#             new_prediction = model.predict(perturbed_row)[0]
#             new_proba = model.predict_proba(perturbed_row)[0]
            
#             if new_prediction != original_prediction:
#                 # Success! Create counterfactual record
#                 return {
#                     'counterfactual_id': cf_id,
#                     'iteration_found': iteration + 1,
#                     'strategy': f"Perturb {len(features_to_perturb)} features: {', '.join(features_to_perturb)}",
#                     'changed_features_list': features_to_perturb,
#                     'num_changed_features': len(features_to_perturb),
#                     'original_prediction': original_prediction,
#                     'new_prediction': new_prediction,
#                     'prediction_changed': True,
#                     'original_proba_class_1': original_proba[1] if len(original_proba) > 1 else original_proba[0],
#                     'new_proba_class_1': new_proba[1] if len(new_proba) > 1 else new_proba[0],
#                     'perturbations': perturbations,
#                     'feature_changes_summary': create_feature_change_summary(perturbations)
#                 }
#         except Exception as e:
#             continue  # Try next iteration
    
#     return None  # Failed to find counterfactual


# def find_counterfactual_single_feature_systematic(model, random_row, feature, original_prediction, cf_id):
#     """Systematically try different perturbation levels for a single feature."""
    
#     original_values = random_row.iloc[0].copy()
#     original_val = original_values[feature]
    
#     # Try different perturbation levels systematically
#     perturbation_levels = [0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 1.0, 1.5, 2.0, 3.0]
#     directions = [1, -1]  # positive and negative perturbations
    
#     for level in perturbation_levels:
#         for direction in directions:
#             perturbed_row = random_row.copy()
#             noise = original_val * level * direction
#             new_val = original_val + noise
            
#             perturbed_row.loc[perturbed_row.index[0], feature] = new_val
            
#             try:
#                 new_prediction = model.predict(perturbed_row)[0]
#                 new_proba = model.predict_proba(perturbed_row)[0]
                
#                 if new_prediction != original_prediction:
#                     perturbations = {
#                         feature: {
#                             'original': original_val,
#                             'new': new_val,
#                             'change': noise,
#                             'percent_change': (noise / original_val * 100) if original_val != 0 else float('inf')
#                         }
#                     }
                    
#                     return {
#                         'counterfactual_id': cf_id,
#                         'iteration_found': len(perturbation_levels) * directions.index(direction) + perturbation_levels.index(level) + 1,
#                         'strategy': f"Systematic single feature: {feature} (level {level}, direction {direction})",
#                         'changed_features_list': [feature],
#                         'num_changed_features': 1,
#                         'original_prediction': original_prediction,
#                         'new_prediction': new_prediction,
#                         'prediction_changed': True,
#                         'original_proba_class_1': original_proba[1] if len(original_proba) > 1 else original_proba[0],
#                         'new_proba_class_1': new_proba[1] if len(new_proba) > 1 else new_proba[0],
#                         'perturbations': perturbations,
#                         'feature_changes_summary': create_feature_change_summary(perturbations)
#                     }
#             except Exception:
#                 continue
    
#     return None


# def find_counterfactual_aggressive(model, random_row, features, original_prediction, cf_id, max_iterations):
#     """Aggressive perturbation of multiple random features."""
    
#     original_values = random_row.iloc[0].copy()
    
#     for iteration in range(max_iterations // 5):  # Fewer iterations but more aggressive
#         perturbed_row = random_row.copy()
#         perturbations = {}
        
#         # Randomly select 2-4 features to perturb
#         num_features = random.randint(2, min(4, len(features)))
#         selected_features = random.sample(features, num_features)
        
#         for feature in selected_features:
#             original_val = original_values[feature]
            
#             # Aggressive perturbation
#             perturbation_factor = random.uniform(0.3, 1.5)  # 30% to 150% change
#             direction = random.choice([1, -1])
#             noise = original_val * perturbation_factor * direction
#             new_val = original_val + noise
            
#             perturbed_row.loc[perturbed_row.index[0], feature] = new_val
#             perturbations[feature] = {
#                 'original': original_val,
#                 'new': new_val,
#                 'change': noise,
#                 'percent_change': (noise / original_val * 100) if original_val != 0 else float('inf')
#             }
        
#         try:
#             new_prediction = model.predict(perturbed_row)[0]
#             new_proba = model.predict_proba(perturbed_row)[0]
            
#             if new_prediction != original_prediction:
#                 return {
#                     'counterfactual_id': cf_id,
#                     'iteration_found': iteration + 1,
#                     'strategy': f"Aggressive multi-feature: {', '.join(selected_features)}",
#                     'changed_features_list': selected_features,
#                     'num_changed_features': len(selected_features),
#                     'original_prediction': original_prediction,
#                     'new_prediction': new_prediction,
#                     'prediction_changed': True,
#                     'original_proba_class_1': original_proba[1] if len(original_proba) > 1 else original_proba[0],
#                     'new_proba_class_1': new_proba[1] if len(new_proba) > 1 else new_proba[0],
#                     'perturbations': perturbations,
#                     'feature_changes_summary': create_feature_change_summary(perturbations)
#                 }
#         except Exception:
#             continue
    
#     return None


# def create_feature_change_summary(perturbations):
#     """Create a readable summary of feature changes."""
#     summaries = []
#     for feature, changes in perturbations.items():
#         orig = changes['original']
#         new = changes['new']
#         pct = changes['percent_change']
        
#         orig_fmt = f"{orig:.3f}" if isinstance(orig, (int, float)) else str(orig)
#         new_fmt = f"{new:.3f}" if isinstance(new, (int, float)) else str(new)
#         pct_fmt = f"{pct:+.1f}%" if isinstance(pct, (int, float)) and np.isfinite(pct) else "inf%"
        
#         summaries.append(f"{feature}: {orig_fmt} → {new_fmt} ({pct_fmt})")
    
#     return "; ".join(summaries)


# def display_counterfactual_results(results_df):
#     """Display counterfactual results in a readable format."""
#     if results_df.empty:
#         print("No results to display.")
#         return
    
#     print("\n🎯 COUNTERFACTUAL RESULTS:")
#     print("=" * 80)
    
#     for _, row in results_df.iterrows():
#         print(f"\n📋 Counterfactual {row['counterfactual_id']}:")
#         print(f"   Strategy: {row['strategy']}")
#         print(f"   Found in {row['iteration_found']} iterations")
#         print(f"   Prediction flip: {row['original_prediction']} → {row['new_prediction']}")
#         print(f"   Probability change: {row['original_proba_class_1']:.3f} → {row['new_proba_class_1']:.3f}")
#         print(f"   Changed features: {', '.join(row['changed_features_list'])}")
#         print(f"   Changes: {row['feature_changes_summary']}")


# # Example usage:
# # Define the features you want to test
# # user_features = ['volt', 'rotate', 'pressure', 'vibration', 'age']

# user_features = df_shap["feature_name"].to_list()

# # Find counterfactuals that change the prediction
# results_df = generate_prediction_changing_counterfactuals(
#     model=model,
#     random_row=random_row,
#     user_features=user_features,
#     num_counterfactuals=5,
#     max_iterations=1,
#     perturbation_strength=0.1,
#     random_seed=42
# )

# # Display results
# display_counterfactual_results(results_df)

# # Show the DataFrame
# if not results_df.empty:
#     print(f"\n📊 DataFrame with {len(results_df)} successful counterfactuals:")
#     display(results_df[['counterfactual_id', 'strategy', 'changed_features_list', 'original_prediction', 'new_prediction', 'feature_changes_summary']])


<br> <br> <br>

---

## Perturb Only one features at a time 
#### Integration with Causal graph 

## Note

the current method:

✅ Single-feature focus - Changes only one feature at a time<br>
✅ Fast and systematic - Tests specific perturbation levels<br>
✅ Causal guidance integration - Uses VARLiNGAM adjacency matrix<br>
✅ Transparent control - You control exactly which feature changes and by how much<br>


dice_ml approach:

❌ Multi-feature changes - Often changes multiple features simultaneously<br>
❌ Less control - Internal optimization decides perturbation levels<br>
❌ No direct causal integration - Doesn't use your structured_adjacency<br>


<br> <br> <br>

---

In [ ]:
import numpy as np
import pandas as pd
import random
from typing import List, Dict, Tuple, Optional

def extract_causal_guidance(structured_adjacency: List[Dict], target_features: List[str]) -> Dict:
    """
    Extract causal guidance for target features from structured adjacency matrix.
    
    Args:
        structured_adjacency: List of dicts with 'source_feature', 'target_feature', 'effect_strength'
        target_features: List of feature names we want guidance for
        
    Returns:
        Dict with causal information for each feature
    """
    causal_info = {}
    
    for edge in structured_adjacency:
        source = edge['source_feature']
        target = edge['target_feature']
        strength = edge['effect_strength']
        
        # Look for direct causal effects TO the target variable
        if target == 'target' and source in target_features:
            causal_info[source] = {
                'effect_strength': abs(strength),
                'effect_direction': 1 if strength > 0 else -1,
                'relationship_type': 'direct_to_target',
                'preferred_direction': 'increase' if strength > 0 else 'decrease'
            }
        
        # Look for causal relationships between features (indirect effects)
        elif source in target_features and target in target_features:
            if source not in causal_info:
                causal_info[source] = {
                    'effect_strength': abs(strength),
                    'effect_direction': 1 if strength > 0 else -1,
                    'relationship_type': 'inter_feature',
                    'preferred_direction': 'increase' if strength > 0 else 'decrease',
                    'affects_feature': target
                }
    
    return causal_info


def prioritize_features_causally(user_features: List[str], structured_adjacency: List[Dict]) -> List[Dict]:
    """
    Prioritize features based on their causal importance to the target.
    
    Args:
        user_features: List of SHAP important features
        structured_adjacency: Causal graph edges
        
    Returns:
        List of feature dicts sorted by causal priority
    """
    feature_priority = []
    
    # Create mapping of direct causal effects to target
    direct_effects = {}
    for edge in structured_adjacency:
        if edge['target_feature'] == 'target' and edge['source_feature'] in user_features:
            direct_effects[edge['source_feature']] = abs(edge['effect_strength'])
    
    # Assign priorities
    for i, feature in enumerate(user_features):
        if feature in direct_effects:
            # Direct causal effect - higher priority
            causal_strength = direct_effects[feature]
            priority_score = causal_strength * 10  # Boost direct effects
        else:
            # No direct causal effect found
            priority_score = 0
        
        feature_priority.append({
            'feature': feature,
            'causal_priority': priority_score,
            'shap_rank': i + 1,
            'has_causal_effect': feature in direct_effects
        })
    
    # Sort by causal priority (descending), then by SHAP rank (ascending)
    feature_priority.sort(key=lambda x: (-x['causal_priority'], x['shap_rank']))
    
    return feature_priority


def get_causal_perturbation_strategy(causal_guidance: Dict, perturbation_levels: List[float], 
                                   original_prediction: int) -> Tuple[List[int], List[float]]:
    """
    Determine optimal perturbation strategy based on causal guidance.
    
    Args:
        causal_guidance: Causal info for the feature
        perturbation_levels: Available perturbation levels
        original_prediction: Current model prediction
        
    Returns:
        Tuple of (directions, levels) to try
    """
    effect_direction = causal_guidance.get('effect_direction', 1)
    effect_strength = causal_guidance.get('effect_strength', 0)
    
    # Determine target prediction (what we want to achieve)
    target_prediction = 1 - original_prediction
    
    # Determine optimal direction based on causal effect
    if target_prediction == 1 and effect_direction > 0:
        # Want prediction=1, feature has positive effect → increase feature
        directions = [1, -1]  # Try increase first
    elif target_prediction == 1 and effect_direction < 0:
        # Want prediction=1, feature has negative effect → decrease feature
        directions = [-1, 1]  # Try decrease first
    elif target_prediction == 0 and effect_direction > 0:
        # Want prediction=0, feature has positive effect → decrease feature
        directions = [-1, 1]  # Try decrease first
    else:
        # Want prediction=0, feature has negative effect → increase feature
        directions = [1, -1]  # Try increase first
    
    # Adjust perturbation levels based on effect strength
    if effect_strength > 0.8:  # Very strong causal effect
        levels = [l for l in perturbation_levels if l <= 0.5]  # Small perturbations
    elif effect_strength > 0.5:  # Strong causal effect
        levels = [l for l in perturbation_levels if l <= 1.0]  # Medium perturbations
    elif effect_strength > 0.2:  # Moderate causal effect
        levels = perturbation_levels  # All levels
    else:  # Weak causal effect
        levels = [l for l in perturbation_levels if l >= 0.1]  # Larger perturbations
    
    return directions, levels


def check_causal_alignment(causal_guidance: Dict, perturbation_direction: int, 
                         original_prediction: int) -> str:
    """
    Check if the successful perturbation aligns with causal expectations.
    
    Args:
        causal_guidance: Causal info for the feature
        perturbation_direction: Direction of perturbation (+1 or -1)
        original_prediction: Original model prediction
        
    Returns:
        Alignment status: 'aligned', 'contrary', or 'no_causal_info'
    """
    if not causal_guidance:
        return "no_causal_info"
    
    effect_direction = causal_guidance.get('effect_direction', 1)
    target_prediction = 1 - original_prediction
    
    # Calculate expected perturbation direction
    if target_prediction == 1 and effect_direction > 0:
        expected_direction = 1  # Should increase feature
    elif target_prediction == 1 and effect_direction < 0:
        expected_direction = -1  # Should decrease feature
    elif target_prediction == 0 and effect_direction > 0:
        expected_direction = -1  # Should decrease feature
    else:
        expected_direction = 1  # Should increase feature
    
    if perturbation_direction == expected_direction:
        return "aligned"
    else:
        return "contrary"


def generate_single_feature_counterfactuals(model, random_row: pd.DataFrame, user_features: List[str],
                                          structured_adjacency: Optional[List[Dict]] = None,
                                          perturbation_levels: Optional[List[float]] = None,
                                          use_causal_guidance: bool = True,
                                          random_seed: int = 42) -> pd.DataFrame:
    """
    Generate counterfactuals by perturbing ONE feature at a time with causal guidance.
    
    Args:
        model: Trained ML model
        random_row: Single row DataFrame to generate counterfactuals for
        user_features: List of SHAP important features
        structured_adjacency: List of causal relationships (optional)
        perturbation_levels: List of perturbation levels as fractions
        use_causal_guidance: Whether to use causal information
        random_seed: Random seed for reproducibility
        
    Returns:
        DataFrame with successful counterfactuals
    """
    print("🎯 Generating Single-Feature Counterfactuals with Causal Guidance")
    print("=" * 70)
    
    # Set random seeds
    np.random.seed(random_seed)
    random.seed(random_seed)
    
    # Default perturbation levels
    if perturbation_levels is None:
        perturbation_levels = [0.05, 0.1, 0.2, 0.3, 0.5, 0.8, 1.0, 1.5, 2.0]
    
    # Validate features
    available_features = set(random_row.columns)
    valid_user_features = [f for f in user_features if f in available_features]
    invalid_features = [f for f in user_features if f not in available_features]
    
    if invalid_features:
        print(f"⚠️ Warning: Features not available: {invalid_features}")
    
    if not valid_user_features:
        print("❌ Error: No valid features provided.")
        return pd.DataFrame()
    
    print(f"✅ Testing {len(valid_user_features)} features")
    
    # Extract causal guidance
    causal_info = {}
    if structured_adjacency and use_causal_guidance:
        print("🔗 Extracting causal guidance...")
        causal_info = extract_causal_guidance(structured_adjacency, valid_user_features)
        
        if causal_info:
            print(f"✅ Found causal guidance for {len(causal_info)} features")
            for feature, info in causal_info.items():
                print(f"   {feature}: effect={info['effect_strength']:.3f}, direction={info['preferred_direction']}")
        else:
            print("⚠️ No direct causal relationships to target found")
    
    # Get baseline prediction
    original_prediction = model.predict(random_row)[0]
    original_proba = model.predict_proba(random_row)[0]
    original_values = random_row.iloc[0].copy()
    
    print(f"\n📊 Original prediction: {original_prediction}")
    print(f"📊 Original probability: {original_proba}")
    print(f"🎯 Target prediction: {1 - original_prediction}")
    
    # Prioritize features
    if causal_info and use_causal_guidance:
        feature_priority = prioritize_features_causally(valid_user_features, structured_adjacency)
        print(f"\n🔗 Causal-guided feature order: {[f['feature'] for f in feature_priority[:5]]}...")
    else:
        feature_priority = [{'feature': f, 'causal_priority': 0, 'shap_rank': i+1} 
                          for i, f in enumerate(valid_user_features)]
        print(f"\n📊 SHAP-guided feature order: {[f['feature'] for f in feature_priority[:5]]}...")
    
    successful_counterfactuals = []
    
    # Test each feature
    for feature_info in feature_priority:
        feature = feature_info['feature']
        shap_rank = feature_info['shap_rank']
        
        print(f"\n🔍 Testing feature #{shap_rank}: {feature}")
        
        original_val = original_values[feature]
        print(f"   Original value: {original_val:.4f}")
        
        # Get causal guidance for this feature
        causal_guidance = causal_info.get(feature, {})
        
        # Determine perturbation strategy
        if causal_guidance and use_causal_guidance:
            directions, levels = get_causal_perturbation_strategy(
                causal_guidance, perturbation_levels, original_prediction
            )
            strategy_type = "causal-guided"
            print(f"   🔗 Causal strategy: {causal_guidance.get('preferred_direction', 'unknown')} first")
        else:
            directions = [1, -1]  # Both directions
            levels = perturbation_levels
            strategy_type = "standard"
            print(f"   📊 Standard strategy: both directions")
        
        # Try perturbations
        found_counterfactual = False
        
        for direction in directions:
            if found_counterfactual:
                break
                
            direction_name = "increase" if direction == 1 else "decrease"
            
            for level in levels:
                if found_counterfactual:
                    break
                
                # Calculate new value
                change = original_val * level * direction
                new_val = original_val + change
                
                # Create perturbed row
                perturbed_row = random_row.copy()
                perturbed_row.loc[perturbed_row.index[0], feature] = new_val
                
                try:
                    # Check prediction
                    new_prediction = model.predict(perturbed_row)[0]
                    new_proba = model.predict_proba(perturbed_row)[0]
                    
                    if new_prediction != original_prediction:
                        # Success! Create counterfactual record
                        percent_change = (change / original_val * 100) if original_val != 0 else float('inf')
                        
                        counterfactual = {
                            'counterfactual_id': len(successful_counterfactuals),
                            'feature_changed': feature,
                            'shap_rank': shap_rank,
                            'causal_priority': feature_info.get('causal_priority', 0),
                            'strategy_used': strategy_type,
                            'direction': direction_name,
                            'perturbation_level': level,
                            'original_value': original_val,
                            'new_value': new_val,
                            'absolute_change': change,
                            'percent_change': percent_change,
                            'original_prediction': original_prediction,
                            'new_prediction': new_prediction,
                            'original_prob_0': original_proba[0] if len(original_proba) > 1 else 1 - original_proba[0],
                            'original_prob_1': original_proba[1] if len(original_proba) > 1 else original_proba[0],
                            'new_prob_0': new_proba[0] if len(new_proba) > 1 else 1 - new_proba[0],
                            'new_prob_1': new_proba[1] if len(new_proba) > 1 else new_proba[0],
                            'probability_change': (new_proba[1] - original_proba[1]) if len(new_proba) > 1 else (new_proba[0] - original_proba[0]),
                            'causal_effect_strength': causal_guidance.get('effect_strength', None),
                            'causal_alignment': check_causal_alignment(causal_guidance, direction, original_prediction) if causal_guidance else None
                        }
                        
                        successful_counterfactuals.append(counterfactual)
                        found_counterfactual = True
                        
                        # Print success
                        print(f"   ✅ SUCCESS! {direction_name} by {level*100:.1f}% → prediction flip")
                        print(f"      Value: {original_val:.4f} → {new_val:.4f} ({percent_change:+.1f}%)")
                        print(f"      Prediction: {original_prediction} → {new_prediction}")
                        print(f"      Probability: {original_proba[1] if len(original_proba) > 1 else original_proba[0]:.3f} → {new_proba[1] if len(new_proba) > 1 else new_proba[0]:.3f}")
                        
                        if causal_guidance:
                            alignment = counterfactual['causal_alignment']
                            print(f"      Causal alignment: {alignment}")
                        
                        break
                        
                except Exception:
                    continue
        
        if not found_counterfactual:
            print(f"   ❌ No prediction flip found for {feature}")
    
    # Create results DataFrame
    if successful_counterfactuals:
        results_df = pd.DataFrame(successful_counterfactuals)
        
        print(f"\n🎉 SUMMARY: Found {len(results_df)} successful counterfactuals!")
        print("=" * 70)
        
        # Analyze causal vs standard strategies
        if use_causal_guidance and causal_info:
            causal_guided = results_df[results_df['strategy_used'] == 'causal-guided']
            standard = results_df[results_df['strategy_used'] == 'standard']
            
            print(f"🔗 Causal-guided successes: {len(causal_guided)}")
            print(f"📊 Standard approach successes: {len(standard)}")
            
            if len(causal_guided) > 0:
                avg_causal_perturbation = causal_guided['perturbation_level'].mean()
                print(f"   Avg perturbation (causal): {avg_causal_perturbation:.3f}")
                
                aligned = causal_guided[causal_guided['causal_alignment'] == 'aligned']
                print(f"   Causally aligned: {len(aligned)}/{len(causal_guided)}")
            
            if len(standard) > 0:
                avg_standard_perturbation = standard['perturbation_level'].mean()
                print(f"   Avg perturbation (standard): {avg_standard_perturbation:.3f}")
        
        # Show detailed results
        print(f"\n📋 DETAILED RESULTS:")
        for _, row in results_df.iterrows():
            feature = row['feature_changed']
            direction = row['direction']
            level = row['perturbation_level']
            rank = row['shap_rank']
            strategy = row['strategy_used']
            
            print(f"✅ #{rank}: {feature} ({strategy})")
            print(f"   Method: {direction} by {level*100:.1f}%")
            print(f"   Change: {row['original_value']:.4f} → {row['new_value']:.4f} ({row['percent_change']:+.1f}%)")
            print(f"   Prediction: {row['original_prediction']} → {row['new_prediction']}")
            
            if row['causal_alignment']:
                print(f"   Causal: {row['causal_alignment']} (strength: {row['causal_effect_strength']:.3f})")
            print()
        
        return results_df
    
    else:
        print(f"\n❌ No counterfactuals found.")
        print("   Suggestions:")
        print("   - Increase perturbation levels: [0.1, 0.5, 1.0, 2.0, 5.0]")
        print("   - Try different features")
        print("   - Check if model is too stable")
        return pd.DataFrame()


def analyze_counterfactual_effectiveness(results_df: pd.DataFrame) -> pd.DataFrame:
    """
    Analyze the effectiveness of different features for counterfactual generation.
    
    Args:
        results_df: Results from generate_single_feature_counterfactuals
        
    Returns:
        Analysis summary DataFrame
    """
    if results_df.empty:
        return pd.DataFrame()
    
    print("📊 COUNTERFACTUAL EFFECTIVENESS ANALYSIS")
    print("=" * 50)
    
    analysis_data = []
    
    for _, row in results_df.iterrows():
        analysis_data.append({
            'feature': row['feature_changed'],
            'shap_rank': row['shap_rank'],
            'strategy_used': row['strategy_used'],
            'perturbation_needed': row['perturbation_level'],
            'percent_change_needed': abs(row['percent_change']),
            'direction_needed': row['direction'],
            'causal_alignment': row.get('causal_alignment', 'unknown'),
            'causal_strength': row.get('causal_effect_strength', 0),
            'probability_change_magnitude': abs(row['probability_change'])
        })
    
    analysis_df = pd.DataFrame(analysis_data)
    
    # Sort by easiest to flip (smallest perturbation needed)
    analysis_df = analysis_df.sort_values('perturbation_needed')
    
    print("🏆 Features ranked by EASE OF FLIPPING (smallest perturbation first):")
    for i, (_, row) in enumerate(analysis_df.iterrows(), 1):
        feature = row['feature']
        rank = row['shap_rank']
        strategy = row['strategy_used']
        level = row['perturbation_needed']
        direction = row['direction_needed']
        alignment = row['causal_alignment']
        
        print(f"{i}. {feature} (SHAP rank #{rank}, {strategy})")
        print(f"   Needs {level*100:.1f}% {direction}")
        print(f"   Causal alignment: {alignment}")
        print(f"   Probability change: {row['probability_change_magnitude']:.3f}")
        print()
    
    return analysis_df


# Example usage:
if __name__ == "__main__":
    # Your usage would be:
    results_df = generate_single_feature_counterfactuals(
        model=model,
        random_row=random_row,
        user_features=df_shap["feature_name"].to_list(),
        structured_adjacency=structured_adjacency,
        use_causal_guidance=True,
        perturbation_levels=[0.05, 0.1, 0.2, 0.3, 0.5, 1.0, 2.0],
        random_seed=42
    )
    
    if not results_df.empty:
        analysis_df = analyze_counterfactual_effectiveness(results_df)
        print("\n📊 Final Analysis:")
        display(analysis_df)



**Key improvements with your `structured_adjacency` structure:**

1. **Proper parsing**: Correctly extracts `source_feature`, `target_feature`, and `effect_strength`
2. **Direct target effects**: Prioritizes features that directly affect the `target` variable
3. **Smart perturbation**: Uses causal effect strength to determine perturbation levels
4. **Causal alignment**: Validates if successful perturbations align with causal expectations
5. **Comprehensive analysis**: Provides detailed effectiveness analysis

**Usage:**


In [ ]:
results_df = generate_single_feature_counterfactuals(
    model=model,
    random_row=random_row,
    user_features=df_shap["feature_name"].to_list(),
    structured_adjacency=structured_adjacency,  # Your causal graph
    use_causal_guidance=True,
    random_seed=42
)

results_df



This will leverage your causal structure to make counterfactual generation more intelligent and interpretable!

<br> <br>

## Helper explanation, step by step**, 


---

# 🔹 1. Setup

**Explanation**:
The function initializes seeds, validates features, and sets default perturbation levels. This ensures reproducibility and avoids invalid features.

**Code**:

```python
np.random.seed(random_seed)
random.seed(random_seed)

# Default perturbation levels if not provided
if perturbation_levels is None:
    perturbation_levels = [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5, 0.6, 0.8, 1.0, 1.5, 2.0]

# Validate user features
available_features = set(random_row.columns)
valid_user_features = [f for f in user_features if f in available_features]
invalid_features = [f for f in user_features if f not in available_features]
```

---

# 🔹 2. Establish Baseline

**Explanation**:
Before perturbing anything, the model’s original prediction and probability are recorded. These act as the reference for counterfactual comparison.

**Code**:

```python
# Get original prediction
original_prediction = model.predict(random_row)[0]
original_proba = model.predict_proba(random_row)[0]
original_values = random_row.iloc[0].copy()

print(f"📊 Original prediction: {original_prediction}")
print(f"📊 Original probability: {original_proba}")
print(f"🎯 Looking for prediction flip to: {1 - original_prediction}")
```

---

# 🔹 3. Extract Causal Guidance from `structured_adjacency`

**Explanation**:
If a causal graph is provided, this code parses it into a dictionary (`causal_info`) with effect strength, direction, and preferred perturbation.

**Code**:

```python
# Parse causal information if provided
causal_info = {}
if structured_adjacency and use_causal_guidance:
    print("🔗 Processing causal relationships...")
    causal_info = extract_causal_guidance(structured_adjacency, valid_user_features)
```

And inside `extract_causal_guidance`:

```python
if target == 'target' and source in target_features:
    causal_info[source] = {
        'effect_strength': abs(strength),
        'effect_direction': 1 if strength > 0 else -1,
        'relationship_type': 'direct_to_target',
        'preferred_direction': 'increase' if strength > 0 else 'decrease'
    }
```

---

# 🔹 4. Prioritize Features (Causal vs. SHAP-only)

**Explanation**:
Features are sorted:

* If causal graph exists → direct-to-target features with higher causal strength are tested first.
* Otherwise → SHAP ranking order.

**Code**:

```python
# Reorder features by causal importance if available
if causal_info and use_causal_guidance:
    feature_priority = prioritize_features_by_causality(valid_user_features, causal_info)
else:
    feature_priority = [{'feature': f, 'causal_priority': i} for i, f in enumerate(valid_user_features)]
```

And inside `prioritize_features_by_causality`:

```python
if feature in causal_info:
    causal_strength = causal_info[feature]['effect_strength']
    if relationship_type == 'direct_to_target':
        priority_score = causal_strength * 2  # direct effects weighted higher
    else:
        priority_score = causal_strength
else:
    priority_score = 0
```

---

# 🔹 5. Perturbation Strategy

**Explanation**:
For each feature:

* If causal guidance is available → use `get_causal_perturbation_strategy` (preferred direction + adjusted levels).
* Otherwise → test both directions with all perturbation levels.

**Code**:

```python
if causal_guidance and use_causal_guidance:
    directions, levels = get_causal_perturbation_strategy(causal_guidance, perturbation_levels, original_prediction)
    strategy_type = "causal-guided"
else:
    directions = [1, -1]  # Standard: both directions
    levels = perturbation_levels
    strategy_type = "standard"
```

Inside `get_causal_perturbation_strategy`:

```python
if target_prediction == 1 and effect_direction > 0:
    directions = [1, -1]  # Want to increase → try increasing first
elif target_prediction == 0 and effect_direction > 0:
    directions = [-1, 1]  # Want to decrease → try decreasing first
```

---

# 🔹 6. Perturbing Feature Values

**Explanation**:
This loop perturbs one feature at different levels and checks if prediction flips.

**Code**:

```python
for direction in directions:
    for level in levels:
        # Calculate new value
        change = original_val * level * direction
        new_val = original_val + change

        # Create perturbed row
        perturbed_row = random_row.copy()
        perturbed_row.loc[perturbed_row.index[0], feature] = new_val

        new_prediction = model.predict(perturbed_row)[0]
        new_proba = model.predict_proba(perturbed_row)[0]
```

---

# 🔹 7. Detecting Counterfactuals

**Explanation**:
When the perturbed row flips the prediction, record it as a counterfactual with all metadata.

**Code**:

```python
if new_prediction != original_prediction:
    counterfactual = {
        'feature_changed': feature,
        'direction': direction_name,
        'perturbation_level': level,
        'original_value': original_val,
        'new_value': new_val,
        'absolute_change': change,
        'percent_change': percent_change,
        'original_prediction': original_prediction,
        'new_prediction': new_prediction,
        'causal_effect_strength': causal_guidance.get('effect_strength', None),
        'causal_alignment': check_causal_alignment(causal_guidance, direction, original_prediction) if causal_guidance else None
    }
    successful_counterfactuals.append(counterfactual)
```

---

# 🔹 8. Checking Causal Alignment

**Explanation**:
After success, check if the change aligns with what the causal graph predicts.

**Code**:

```python
def check_causal_alignment(causal_guidance: dict, perturbation_direction: int, original_prediction: int) -> str:
    if target_prediction == 1 and effect_direction > 0:
        expected_direction = 1  # Should increase
    elif target_prediction == 0 and effect_direction > 0:
        expected_direction = -1  # Should decrease
    ...
    return "aligned" if perturbation_direction == expected_direction else "contrary"
```

---

# 🔹 9. Summarizing Results

**Explanation**:
At the end, the function returns all successful counterfactuals and compares causal-guided vs. standard strategies.

**Code**:

```python
if successful_counterfactuals:
    results_df = pd.DataFrame(successful_counterfactuals)
    causal_guided = results_df[results_df['strategy_used'] == 'causal-guided']
    standard = results_df[results_df['strategy_used'] == 'standard']
    return results_df
else:
    return pd.DataFrame()
```

---

✅ So now you have a **step-by-step guide** with the **exact piece of code** for each part of the counterfactual generation process.

Would you like me to **turn this into a visual flow diagram** (with arrows: input row → baseline → perturbation strategy → prediction check → counterfactual) so you can use it for teaching/explaining?
